##Import Libraries

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
from google.colab import drive
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

print("Libraries loaded successfully.")

Libraries loaded successfully.


##Mount Google Drive

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive


drive.mount("/content/gdrive")
# drive.mount("/content/drive")

Mounted at /content/gdrive


##Build and audit the full-cohort patient maste

In [ ]:

# ============================================================
# FULL COHORT CLINICAL-GENOMIC PREPROCESSING
# STEP 1: PATIENT MASTER AND ELIGIBILITY AUDIT
#
# Leakage safeguards:
# - Keep final validation/test patients protected.
# - Match by patient ID, never row position.
# - Do not fit imputers, encoders, scalers, or feature selectors here.
# - Do not overwrite any existing pipeline files.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
BASE = Path("/content/gdrive/MyDrive/TCGA_BRCA")

CLINICAL_SOURCE = Path(
    "/content/gdrive/MyDrive/BreastCanser-DataSet/"
    "Human__TCGA_BRCA__MS__Clinical__Clinical__01_28_2016__BI__Clinical__Firehose.tsv"
)

GENOMIC_SOURCE = Path(
    "/content/gdrive/MyDrive/BreastCanser-DataSet/"
    "Human__TCGA_BRCA__UNC__RNAseq__HiSeq_RNA__01_28_2016__BI__Gene__Firehose_RSEM_log2.tsv"
)

ENDPOINT_PATH = Path(
    "/content/gdrive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

SPLIT_DIR = BASE / "dataset_split_rfs_endpoint_v2"

# A new folder: does not replace existing preprocessing outputs.
OUT_DIR = BASE / "clinical_genomic_full_preprocessing_leakage_free_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Check required files
# ------------------------------------------------------------
required_paths = [
    CLINICAL_SOURCE,
    GENOMIC_SOURCE,
    ENDPOINT_PATH,
    SPLIT_DIR / "train_rfs_patient_ids.csv",
    SPLIT_DIR / "validation_rfs_patient_ids.csv",
    SPLIT_DIR / "test_rfs_patient_ids.csv",
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("All required source files found.")

# ------------------------------------------------------------
# 3. Standardize patient IDs for matching
# ------------------------------------------------------------
def normalize_ids(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(".", "-", regex=False)
        .str.upper()
    )

def load_split_ids(filename):
    df = pd.read_csv(SPLIT_DIR / filename)

    if "patient_id" not in df.columns:
        raise ValueError(f"patient_id missing in {filename}")

    ids = normalize_ids(df["patient_id"]).dropna()

    if ids.duplicated().any():
        raise ValueError(f"Duplicate IDs detected in {filename}")

    return set(ids)

train_ids = load_split_ids("train_rfs_patient_ids.csv")
val_ids = load_split_ids("validation_rfs_patient_ids.csv")
test_ids = load_split_ids("test_rfs_patient_ids.csv")

assert not (train_ids & val_ids), "Train/validation overlap!"
assert not (train_ids & test_ids), "Train/test overlap!"
assert not (val_ids & test_ids), "Validation/test overlap!"

protected_ids = val_ids | test_ids

print("\nProtected final split sizes:")
print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))
print("Validation + test protected:", len(protected_ids))

# ------------------------------------------------------------
# 4. Load raw source files
# ------------------------------------------------------------
clinical_raw = pd.read_csv(
    CLINICAL_SOURCE, sep="\t", low_memory=False
)

genomic_raw = pd.read_csv(
    GENOMIC_SOURCE, sep="\t", low_memory=False
)

endpoint = pd.read_csv(ENDPOINT_PATH)

# ------------------------------------------------------------
# 5. Extract clinical patient IDs
#    Raw clinical source: rows are attributes, columns are patients.
# ------------------------------------------------------------
if "attrib_name" not in clinical_raw.columns:
    raise ValueError(
        "Clinical source does not contain the expected 'attrib_name' column."
    )

clinical_sample_cols = [
    c for c in clinical_raw.columns if c != "attrib_name"
]

clinical_ids = set(
    normalize_ids(pd.Series(clinical_sample_cols)).dropna()
)

# ------------------------------------------------------------
# 6. Extract genomic patient IDs
#    Raw genomic source: rows are genes, columns are patients.
# ------------------------------------------------------------
if "attrib_name" not in genomic_raw.columns:
    raise ValueError(
        "Genomic source does not contain the expected 'attrib_name' column."
    )

genomic_sample_cols = [
    c for c in genomic_raw.columns if c != "attrib_name"
]

genomic_ids = set(
    normalize_ids(pd.Series(genomic_sample_cols)).dropna()
)

# ------------------------------------------------------------
# 7. Validate and normalize the endpoint
# ------------------------------------------------------------
required_endpoint_cols = {
    "patient_id", "survival_time", "event"
}

missing_cols = required_endpoint_cols - set(endpoint.columns)

if missing_cols:
    raise ValueError(
        f"Endpoint is missing required columns: {sorted(missing_cols)}"
    )

endpoint["patient_id"] = normalize_ids(endpoint["patient_id"])
endpoint["survival_time"] = pd.to_numeric(
    endpoint["survival_time"], errors="coerce"
)
endpoint["event"] = pd.to_numeric(
    endpoint["event"], errors="coerce"
)

if endpoint["patient_id"].duplicated().any():
    raise ValueError("Duplicate patient IDs in endpoint.")

if endpoint["survival_time"].isna().any():
    raise ValueError("Missing/non-numeric survival times in endpoint.")

if (endpoint["survival_time"] <= 0).any():
    raise ValueError("Non-positive survival times in endpoint.")

if not endpoint["event"].isin([0, 1]).all():
    raise ValueError("Endpoint event must contain only 0 or 1.")

endpoint_ids = set(endpoint["patient_id"].dropna())

# ------------------------------------------------------------
# 8. Build the full data-availability intersection
# ------------------------------------------------------------
all_three_ids = clinical_ids & genomic_ids & endpoint_ids

# Final validation and test must never enter pretraining.
pretraining_candidate_ids = all_three_ids - protected_ids

# Existing final training patients are part of the candidate pool.
final_train_in_intersection = train_ids & all_three_ids

# These are additional patients outside all final splits, but
# with clinical, genomic and endpoint records.
additional_candidate_ids = (
    pretraining_candidate_ids - train_ids
)

# IDs in the existing final splits that do not have all three sources.
missing_final_train = train_ids - all_three_ids
missing_final_val = val_ids - all_three_ids
missing_final_test = test_ids - all_three_ids

print("\n" + "=" * 65)
print("PATIENT AVAILABILITY")
print("=" * 65)
print("Clinical source IDs:", len(clinical_ids))
print("Genomic source IDs:", len(genomic_ids))
print("Audited endpoint IDs:", len(endpoint_ids))
print("Patients in all three sources:", len(all_three_ids))

print("\n" + "=" * 65)
print("FINAL SPLIT COVERAGE")
print("=" * 65)
print("Final train IDs present in all sources:", len(final_train_in_intersection))
print("Final train IDs missing at least one source:", len(missing_final_train))
print("Final validation IDs missing at least one source:", len(missing_final_val))
print("Final test IDs missing at least one source:", len(missing_final_test))

print("\n" + "=" * 65)
print("PRETRAINING CANDIDATE POOL")
print("=" * 65)
print("All eligible IDs excluding final validation/test:",
      len(pretraining_candidate_ids))
print("Existing final train IDs in candidate pool:",
      len(final_train_in_intersection))
print("Additional candidate IDs outside final splits:",
      len(additional_candidate_ids))

# ------------------------------------------------------------
# 9. Create a patient-level master table
# ------------------------------------------------------------
clinical_only_ids = clinical_ids - genomic_ids - endpoint_ids
genomic_only_ids = genomic_ids - clinical_ids - endpoint_ids
endpoint_only_ids = endpoint_ids - clinical_ids - genomic_ids

master_ids = (
    clinical_ids | genomic_ids | endpoint_ids
    | train_ids | val_ids | test_ids
)

master = pd.DataFrame({
    "patient_id": sorted(master_ids)
})

master["has_clinical"] = master["patient_id"].isin(clinical_ids)
master["has_genomic"] = master["patient_id"].isin(genomic_ids)
master["has_rfs_endpoint"] = master["patient_id"].isin(endpoint_ids)
master["in_final_train"] = master["patient_id"].isin(train_ids)
master["in_final_validation"] = master["patient_id"].isin(val_ids)
master["in_final_test"] = master["patient_id"].isin(test_ids)
master["has_all_three_sources"] = master[
    ["has_clinical", "has_genomic", "has_rfs_endpoint"]
].all(axis=1)

def assign_group(patient_id):
    if patient_id in val_ids:
        return "PROTECTED_FINAL_VALIDATION"
    if patient_id in test_ids:
        return "PROTECTED_FINAL_TEST"
    if patient_id in train_ids:
        return "FINAL_TRAIN"
    if patient_id in all_three_ids:
        return "ADDITIONAL_PRETRAINING_CANDIDATE"
    return "NOT_ELIGIBLE_OR_INCOMPLETE"

master["cohort_group"] = master["patient_id"].map(assign_group)

# Add endpoint targets by patient ID, without positional matching.
master = master.merge(
    endpoint[["patient_id", "survival_time", "event"]],
    on="patient_id",
    how="left",
    validate="one_to_one"
)

# A candidate eligibility flag, not a claim that training has occurred.
master["eligible_for_cg_pretraining_pool"] = (
    master["has_all_three_sources"]
    & ~master["in_final_validation"]
    & ~master["in_final_test"]
)

# ------------------------------------------------------------
# 10. Summarize event counts for each group
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("COHORT GROUP SUMMARY")
print("=" * 65)

summary = (
    master.groupby("cohort_group", dropna=False)
    .agg(
        patients=("patient_id", "nunique"),
        endpoint_patients=("event", "count"),
        events=("event", lambda x: int((x == 1).sum())),
        censored=("event", lambda x: int((x == 0).sum()))
    )
    .reset_index()
)

print(summary.to_string(index=False))

# ------------------------------------------------------------
# 11. Save new audit outputs only
# ------------------------------------------------------------
master_path = OUT_DIR / "full_cohort_patient_master_audit.csv"
summary_path = OUT_DIR / "full_cohort_group_summary.csv"

master.to_csv(master_path, index=False)
summary.to_csv(summary_path, index=False)

print("\nNew files saved:")
print(master_path)
print(summary_path)

print("\nNo existing clinical, genomic, split, endpoint, or pathology files were modified.")


All required source files found.

Protected final split sizes:
Train: 236
Validation: 49
Test: 52
Validation + test protected: 101

PATIENT AVAILABILITY
Clinical source IDs: 1097
Genomic source IDs: 1093
Audited endpoint IDs: 1062
Patients in all three sources: 1059

FINAL SPLIT COVERAGE
Final train IDs present in all sources: 236
Final train IDs missing at least one source: 0
Final validation IDs missing at least one source: 0
Final test IDs missing at least one source: 0

PRETRAINING CANDIDATE POOL
All eligible IDs excluding final validation/test: 958
Existing final train IDs in candidate pool: 236
Additional candidate IDs outside final splits: 722

COHORT GROUP SUMMARY
                    cohort_group  patients  endpoint_patients  events  censored
ADDITIONAL_PRETRAINING_CANDIDATE       722                722       5       717
                     FINAL_TRAIN       236                236      54       182
      NOT_ELIGIBLE_OR_INCOMPLETE        38                  3       0         3

##Test Cell

In [ ]:

from pathlib import Path
import pandas as pd

BASE = Path("/content/gdrive/MyDrive/TCGA_BRCA")

FEATURE_DIR = (
    BASE / "rfs_feature_selection_leakage_free_endpoint_v2"
)
CLINICAL_DIR = BASE / "clinical_preprocessing_leakage_free"
AUDIT_DIR = BASE / "clinical_genomic_full_preprocessing_leakage_free_v1"

print("=" * 70)
print("1. GENOMIC PREPROCESSING ARTIFACTS")
print("=" * 70)

for path in sorted(FEATURE_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(FEATURE_DIR))

print("\n" + "=" * 70)
print("2. SELECTED GENE LIST")
print("=" * 70)

gene_file = FEATURE_DIR / "rfs_selected_genes_200.csv"

if gene_file.exists():
    genes_df = pd.read_csv(gene_file)
    print("Shape:", genes_df.shape)
    print("Columns:", genes_df.columns.tolist())
    print(genes_df.head(10).to_string(index=False))
else:
    print("Selected-gene file not found at:", gene_file)

print("\n" + "=" * 70)
print("3. CLINICAL PREPROCESSING ARTIFACTS")
print("=" * 70)

if CLINICAL_DIR.exists():
    for path in sorted(CLINICAL_DIR.rglob("*")):
        if path.is_file():
            print(path.relative_to(CLINICAL_DIR))
else:
    print("Clinical preprocessing directory not found.")

print("\n" + "=" * 70)
print("4. CURRENT COHORT AUDIT FILES")
print("=" * 70)

for filename in [
    "full_cohort_patient_master_audit.csv",
    "full_cohort_group_summary.csv",
]:
    path = AUDIT_DIR / filename
    print(f"\n{filename}: exists={path.exists()}")

    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))

print("\n" + "=" * 70)
print("5. EXISTING FINAL TRAINING FEATURE FILES")
print("=" * 70)

for path in [
    FEATURE_DIR / "genomic_train_200_rfs_leakage_free.csv",
    CLINICAL_DIR / "clinical_train_leakage_free.csv",
]:
    print(f"\nFile: {path}")
    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("First columns:", df.columns[:8].tolist())
        print("Last columns:", df.columns[-5:].tolist())
    else:
        print("NOT FOUND")

print("\nRead-only inspection completed. No files modified.")


1. GENOMIC PREPROCESSING ARTIFACTS
genomic_standard_scaler_200.pkl
genomic_test_200_rfs_leakage_free.csv
genomic_train_200_rfs_leakage_free.csv
genomic_validation_200_rfs_leakage_free.csv
rfs_selected_genes_200.csv

2. SELECTED GENE LIST
Shape: (200, 1)
Columns: ['rfs_selected_genes']
rfs_selected_genes
              PSCA
           ZNF385B
            MYBPC1
            CALML5
             PRAME
              FLT3
             GREB1
            TSPAN8
             IL6ST
            TUBA3E

3. CLINICAL PREPROCESSING ARTIFACTS
clinical_test_leakage_free.csv
clinical_train_leakage_free.csv
clinical_validation_leakage_free.csv

4. CURRENT COHORT AUDIT FILES

full_cohort_patient_master_audit.csv: exists=True
Shape: (1097, 12)
Columns: ['patient_id', 'has_clinical', 'has_genomic', 'has_rfs_endpoint', 'in_final_train', 'in_final_validation', 'in_final_test', 'has_all_three_sources', 'cohort_group', 'survival_time', 'event', 'eligible_for_cg_pretraining_pool']
  patient_id  has_clinical  has_

##Diagnose the scaler file and check whether the genomic CSVs are already scaled

In [ ]:

from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import sys

BASE = Path("/content/gdrive/MyDrive/TCGA_BRCA")
GENOMIC_DIR = BASE / "rfs_feature_selection_leakage_free_endpoint_v2"

SCALER_PATH = GENOMIC_DIR / "genomic_standard_scaler_200.pkl"
TRAIN_PATH = GENOMIC_DIR / "genomic_train_200_rfs_leakage_free.csv"

print("=" * 65)
print("1. SCALER FILE INFORMATION")
print("=" * 65)

print("Exists:", SCALER_PATH.exists())
print("Size (bytes):", SCALER_PATH.stat().st_size)

with open(SCALER_PATH, "rb") as f:
    header = f.read(32)

print("First bytes (hex):", header.hex())
print("Python version:", sys.version)

print("\n" + "=" * 65)
print("2. TRY LOADING WITH JOBLIB")
print("=" * 65)

scaler = None

try:
    import joblib
    scaler = joblib.load(SCALER_PATH)
    print("Loaded successfully using joblib.")
    print("Object type:", type(scaler))
    print("Expected features:",
          getattr(scaler, "n_features_in_", "not available"))
except Exception as e:
    print("joblib.load failed:", type(e).__name__, str(e))

print("\n" + "=" * 65)
print("3. CHECK EXISTING GENOMIC TRAINING VALUES")
print("=" * 65)

train = pd.read_csv(TRAIN_PATH)

gene_cols = [
    c for c in train.columns
    if c not in ["patient_id", "survival_time", "event"]
]

print("Training rows:", len(train))
print("Gene columns:", len(gene_cols))
print("Missing gene values:", int(train[gene_cols].isna().sum().sum()))

print("\nFirst 10 gene means and standard deviations:")
stats = pd.DataFrame({
    "mean": train[gene_cols].mean(),
    "std": train[gene_cols].std(ddof=0),
    "min": train[gene_cols].min(),
    "max": train[gene_cols].max(),
})
print(stats.head(10).to_string())

print("\nOverall absolute gene-mean summary:")
print(stats["mean"].abs().describe().to_string())

print("\nOverall gene standard-deviation summary:")
print(stats["std"].describe().to_string())

print("\n" + "=" * 65)
print("4. CHECK EXPECTED FEATURE ORDER")
print("=" * 65)

gene_file = GENOMIC_DIR / "rfs_selected_genes_200.csv"
selected_genes = pd.read_csv(gene_file)["rfs_selected_genes"].astype(str).tolist()

print("Selected genes:", len(selected_genes))
print("Training gene columns match selected gene order:",
      gene_cols == selected_genes)

print("\nDiagnostic finished. No files modified.")


1. SCALER FILE INFORMATION
Exists: True
Size (bytes): 7367
First bytes (hex): 8004953d010000000000008c1b736b6c6561726e2e70726570726f6365737369
Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

2. TRY LOADING WITH JOBLIB
Loaded successfully using joblib.
Object type: <class 'sklearn.preprocessing._data.StandardScaler'>
Expected features: 200

3. CHECK EXISTING GENOMIC TRAINING VALUES
Training rows: 236
Gene columns: 200
Missing gene values: 0

First 10 gene means and standard deviations:
                 mean  std       min       max
PSCA    -1.411300e-16  1.0 -1.852559  3.374091
ZNF385B  6.962416e-17  1.0 -1.955888  2.454662
MYBPC1   2.069907e-17  1.0 -1.276973  3.046923
CALML5   3.462390e-16  1.0 -1.801411  2.268252
PRAME   -1.909960e-16  1.0 -1.192736  2.133539
FLT3     3.989276e-16  1.0 -2.144389  1.955930
GREB1    7.526936e-16  1.0 -2.906383  1.543058
TSPAN8  -5.645202e-17  1.0 -1.062788  3.517599
IL6ST    5.268855e-16  1.0 -4.212579  2.530233
TUBA3E   1.975821e-1

##Create new files

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.preprocessing import StandardScaler

# ============================================================
# FULL CLINICAL-GENOMIC PREPROCESSING — LEAKAGE-FREE V1
# ============================================================
# Fits clinical preprocessing on final train IDs only.
# Reuses the saved genomic scaler fitted on final training data.
# Never includes final validation/test patients in the CG pool.
# Never modifies existing source or feature files.
# ============================================================

BASE = Path("/content/gdrive/MyDrive/TCGA_BRCA")
RAW_DIR = Path("/content/gdrive/MyDrive/BreastCanser-DataSet")

CLINICAL_SOURCE = RAW_DIR / (
    "Human__TCGA_BRCA__MS__Clinical__Clinical__01_28_2016__BI__Clinical__Firehose.tsv"
)
GENOMIC_SOURCE = RAW_DIR / (
    "Human__TCGA_BRCA__UNC__RNAseq__HiSeq_RNA__01_28_2016__BI__Gene__Firehose_RSEM_log2.tsv"
)
ENDPOINT_PATH = Path(
    "/content/gdrive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

SPLIT_DIR = BASE / "dataset_split_rfs_endpoint_v2"
GENOMIC_DIR = BASE / "rfs_feature_selection_leakage_free_endpoint_v2"
MASTER_PATH = (
    BASE / "clinical_genomic_full_preprocessing_leakage_free_v1"
    / "full_cohort_patient_master_audit.csv"
)
GENE_LIST_PATH = GENOMIC_DIR / "rfs_selected_genes_200.csv"
GENOMIC_SCALER_PATH = GENOMIC_DIR / "genomic_standard_scaler_200.pkl"

# All newly created features go here.
OUT_DIR = (
    BASE / "clinical_genomic_full_preprocessing_leakage_free_v1"
    / "preprocessed_features_v1"
)

# Prevent accidental overwriting on reruns.
if OUT_DIR.exists() and any(OUT_DIR.iterdir()):
    raise FileExistsError(
        f"Output folder already contains files:\n{OUT_DIR}\n"
        "Choose a new output folder name before rerunning this cell."
    )

for path in [
    CLINICAL_SOURCE, GENOMIC_SOURCE, ENDPOINT_PATH,
    MASTER_PATH, GENE_LIST_PATH, GENOMIC_SCALER_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

# ------------------------------------------------------------
# 1. Helpers and final protected splits
# ------------------------------------------------------------
def normalize_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(".", "-", regex=False)
        .str.upper()
    )

def load_split(filename):
    df = pd.read_csv(SPLIT_DIR / filename)
    if "patient_id" not in df.columns:
        raise ValueError(f"patient_id missing in {filename}")
    ids = normalize_id(df["patient_id"]).dropna()
    if ids.duplicated().any():
        raise ValueError(f"Duplicate IDs in {filename}")
    return set(ids)

train_ids = load_split("train_rfs_patient_ids.csv")
val_ids = load_split("validation_rfs_patient_ids.csv")
test_ids = load_split("test_rfs_patient_ids.csv")

assert len(train_ids) == 236
assert len(val_ids) == 49
assert len(test_ids) == 52
assert not (train_ids & val_ids)
assert not (train_ids & test_ids)
assert not (val_ids & test_ids)

protected_ids = val_ids | test_ids

# ------------------------------------------------------------
# 2. Load cohort audit and audited RFS endpoint
# ------------------------------------------------------------
master = pd.read_csv(MASTER_PATH)
master["patient_id"] = normalize_id(master["patient_id"])

endpoint = pd.read_csv(ENDPOINT_PATH)
endpoint["patient_id"] = normalize_id(endpoint["patient_id"])

if endpoint["patient_id"].duplicated().any():
    raise ValueError("Endpoint contains duplicate patient IDs.")

endpoint["survival_time"] = pd.to_numeric(
    endpoint["survival_time"], errors="coerce"
)
endpoint["event"] = pd.to_numeric(endpoint["event"], errors="coerce")

if endpoint[["survival_time", "event"]].isna().any().any():
    raise ValueError("Endpoint has missing survival_time/event values.")

if not endpoint["event"].isin([0, 1]).all():
    raise ValueError("Event values must be 0 or 1.")

if not (endpoint["survival_time"] > 0).all():
    raise ValueError("Survival times must be positive.")

if master["patient_id"].duplicated().any():
    raise ValueError("Master audit contains duplicate patient IDs.")

master = master.drop(
    columns=["survival_time", "event"], errors="ignore"
).merge(
    endpoint[["patient_id", "survival_time", "event"]],
    on="patient_id", how="left", validate="one_to_one"
)

# Candidate pool: final train + additional eligible candidates.
pool_ids = set(
    master.loc[
        master["eligible_for_cg_pretraining_pool"].eq(True),
        "patient_id"
    ].dropna()
)

assert len(pool_ids) == 958, (
    f"Expected 958 pretraining candidates; found {len(pool_ids)}"
)
assert train_ids.issubset(pool_ids), "Final train IDs missing from pool."
assert not (pool_ids & protected_ids), (
    "LEAKAGE CHECK FAILED: validation/test IDs entered CG pool."
)

additional_ids = pool_ids - train_ids
assert len(additional_ids) == 722

# ------------------------------------------------------------
# 3. Load raw clinical source and orient one row per patient
# ------------------------------------------------------------
clinical_raw = pd.read_csv(
    CLINICAL_SOURCE, sep="\t", low_memory=False
)

if "attrib_name" not in clinical_raw.columns:
    raise ValueError("Clinical source lacks attrib_name.")

clinical_raw = clinical_raw.set_index("attrib_name").T
clinical_raw.index.name = "patient_id"
clinical_raw = clinical_raw.reset_index()
clinical_raw["patient_id"] = normalize_id(clinical_raw["patient_id"])

if clinical_raw["patient_id"].duplicated().any():
    raise ValueError("Duplicate patient IDs in clinical source.")

clinical_columns = [
    "years_to_birth",
    "Tumor_purity",
    "pathologic_stage",
    "pathology_T_stage",
    "pathology_N_stage",
    "pathology_M_stage",
    "histological_type",
    "number_of_lymph_nodes",
    "PAM50",
    "radiation_therapy",
    "race",
    "ethnicity",
]

missing = [c for c in clinical_columns if c not in clinical_raw.columns]
if missing:
    raise ValueError(f"Missing clinical features: {missing}")

clinical_raw = clinical_raw[["patient_id"] + clinical_columns].copy()

# Standardize common missing-value representations.
missing_tokens = [
    "", "NA", "N/A", "NaN", "NAN",
    "[Not Available]", "[Not Applicable]", "--", "null", "NULL"
]

for col in clinical_columns:
    clinical_raw[col] = clinical_raw[col].replace(
        missing_tokens, np.nan
    )

# Match all IDs to the eligible pool and final split patients.
needed_ids = pool_ids | protected_ids

clinical_raw = clinical_raw[
    clinical_raw["patient_id"].isin(needed_ids)
].copy()

if set(train_ids) - set(clinical_raw["patient_id"]):
    raise ValueError("Some final train patients are absent from clinical data.")
if set(val_ids) - set(clinical_raw["patient_id"]):
    raise ValueError("Some validation patients are absent from clinical data.")
if set(test_ids) - set(clinical_raw["patient_id"]):
    raise ValueError("Some test patients are absent from clinical data.")

clinical_raw = clinical_raw.set_index("patient_id").reindex(
    sorted(needed_ids)
)
clinical_raw.index.name = "patient_id"

# ------------------------------------------------------------
# 4. Clinical numeric processing: fit on FINAL TRAIN only
# ------------------------------------------------------------
continuous_cols = [
    "years_to_birth", "Tumor_purity", "number_of_lymph_nodes"
]

ordinal_maps = {
    "pathologic_stage": {
        "stagei": 1, "stageii": 2, "stageiii": 3, "stageiv": 4
    },
    "pathology_T_stage": {
        "t1": 1, "t2": 2, "t3": 3, "t4": 4
    },
    "pathology_N_stage": {
        "n0": 0, "n1": 1, "n2": 2, "n3": 3
    },
    "pathology_M_stage": {
        "m0": 0, "m1": 1
    },
}

numeric_cols = continuous_cols + list(ordinal_maps) + [
    "radiation_therapy"
]

clinical_num = pd.DataFrame(index=clinical_raw.index)

for col in continuous_cols:
    clinical_num[col] = pd.to_numeric(
        clinical_raw[col], errors="coerce"
    )

for col, mapping in ordinal_maps.items():
    values = (
        clinical_raw[col].astype("string")
        .str.strip().str.lower()
        .str.replace(r"\s+", "", regex=True)
    )
    clinical_num[col] = values.map(mapping).astype(float)

radiation = (
    clinical_raw["radiation_therapy"].astype("string")
    .str.strip().str.lower()
)
clinical_num["radiation_therapy"] = radiation.map(
    {"yes": 1.0, "no": 0.0}
)

# Fit median imputations on the 236 final training IDs only.
train_num = clinical_num.loc[sorted(train_ids)]

numeric_medians = {}
for col in numeric_cols:
    median = train_num[col].median()
    if pd.isna(median):
        raise ValueError(
            f"Training data has no usable values for {col}; "
            "do not impute this feature from the full cohort."
        )
    numeric_medians[col] = float(median)
    clinical_num[col] = clinical_num[col].fillna(median)

# Fit standardization only on final train, for continuous variables.
clinical_scaler = StandardScaler()
clinical_scaler.fit(clinical_num.loc[sorted(train_ids), continuous_cols])

clinical_num.loc[:, continuous_cols] = clinical_scaler.transform(
    clinical_num[continuous_cols]
)

# ------------------------------------------------------------
# 5. Clinical categorical processing: fit modes on FINAL TRAIN
# ------------------------------------------------------------
categorical_cols = [
    "histological_type", "PAM50", "race", "ethnicity"
]

clinical_cat = pd.DataFrame(index=clinical_raw.index)

for col in categorical_cols:
    clinical_cat[col] = (
        clinical_raw[col].astype("string")
        .str.strip().str.lower()
        .str.replace(r"\s+", "", regex=True)
    )
    clinical_cat[col] = clinical_cat[col].replace(
        {"<na>": pd.NA, "nan": pd.NA, "none": pd.NA}
    )

# Fixed categories from the established 21-column clinical schema.
category_levels = {
    "histological_type": [
        "infiltratingductalcarcinoma",
        "infiltratinglobularcarcinoma",
        "medullarycarcinoma",
        "metaplasticcarcinoma",
        "mixedhistology(pleasespecify)",
        "mucinouscarcinoma",
        "other,specify",
    ],
    "PAM50": ["basal", "her2", "luma", "lumb"],
    "race": ["asian", "blackorafricanamerican", "white"],
    "ethnicity": ["hispanicorlatino", "nothispanicorlatino"],
}

categorical_modes = {}
for col in categorical_cols:
    mode = clinical_cat.loc[sorted(train_ids), col].mode(dropna=True)
    if mode.empty:
        raise ValueError(
            f"No usable training category for {col}; "
            "do not infer it from validation/test."
        )
    categorical_modes[col] = str(mode.iloc[0])
    clinical_cat[col] = clinical_cat[col].fillna(categorical_modes[col])

    unknown = sorted(
        set(clinical_cat.loc[sorted(needed_ids), col].dropna())
        - set(category_levels[col])
    )
    if unknown:
        raise ValueError(
            f"Unexpected categories in {col}: {unknown}. "
            "Review category mapping before encoding."
        )

# Generate one-hot columns with explicit reference levels and order.
# Reference categories reproduce the established feature schema.
reference_category = {
    "histological_type": "infiltratingductalcarcinoma",
    "PAM50": "basal",
    "race": "asian",
    "ethnicity": "hispanicorlatino",
}

dummy_columns = [
    "histological_type_infiltratinglobularcarcinoma",
    "histological_type_medullarycarcinoma",
    "histological_type_metaplasticcarcinoma",
    "histological_type_mixedhistology(pleasespecify)",
    "histological_type_mucinouscarcinoma",
    "histological_type_other,specify",
    "PAM50_Her2",
    "PAM50_LumA",
    "PAM50_LumB",
    "race_blackorafricanamerican",
    "race_white",
    "ethnicity_nothispanicorlatino",
]

dummy_spec = {
    "histological_type": {
        "infiltratinglobularcarcinoma":
            "histological_type_infiltratinglobularcarcinoma",
        "medullarycarcinoma":
            "histological_type_medullarycarcinoma",
        "metaplasticcarcinoma":
            "histological_type_metaplasticcarcinoma",
        "mixedhistology(pleasespecify)":
            "histological_type_mixedhistology(pleasespecify)",
        "mucinouscarcinoma":
            "histological_type_mucinouscarcinoma",
        "other,specify": "histological_type_other,specify",
    },
    "PAM50": {
        "her2": "PAM50_Her2",
        "luma": "PAM50_LumA",
        "lumb": "PAM50_LumB",
    },
    "race": {
        "blackorafricanamerican": "race_blackorafricanamerican",
        "white": "race_white",
    },
    "ethnicity": {
        "nothispanicorlatino": "ethnicity_nothispanicorlatino",
    },
}

clinical_dummies = pd.DataFrame(
    0.0, index=clinical_raw.index, columns=dummy_columns
)

for col, mapping in dummy_spec.items():
    for category, output_col in mapping.items():
        clinical_dummies[output_col] = (
            clinical_cat[col].eq(category).astype(float)
        )

# Preserve the established clinical feature order.
clinical_feature_order = [
    "years_to_birth",
    "Tumor_purity",
    "pathologic_stage",
    "pathology_T_stage",
    "pathology_N_stage",
    "pathology_M_stage",
    "number_of_lymph_nodes",
    "radiation_therapy",
] + dummy_columns

clinical_features = pd.concat(
    [clinical_num, clinical_dummies], axis=1
)[clinical_feature_order]

if clinical_features.shape[1] != 20:
    raise ValueError(
        f"Expected 20 clinical features, got {clinical_features.shape[1]}"
    )

if clinical_features.isna().any().any():
    raise ValueError("Clinical feature matrix still contains missing values.")

# ------------------------------------------------------------
# 6. Genomic processing: reuse the exact selected genes/scaler
# ------------------------------------------------------------
gene_df = pd.read_csv(GENE_LIST_PATH)
genes = gene_df["rfs_selected_genes"].astype(str).str.strip().tolist()

if len(genes) != 200 or len(set(genes)) != 200:
    raise ValueError("Gene list must contain exactly 200 unique genes.")

genomic_raw = pd.read_csv(
    GENOMIC_SOURCE, sep="\t", low_memory=False
)

if "attrib_name" not in genomic_raw.columns:
    raise ValueError("Genomic source lacks attrib_name.")

genomic_raw["attrib_name"] = (
    genomic_raw["attrib_name"].astype(str).str.strip()
)

if genomic_raw["attrib_name"].duplicated().any():
    raise ValueError(
        "Duplicate gene names in raw genomic source. "
        "Resolve them explicitly before extracting features."
    )

gene_table = genomic_raw.set_index("attrib_name")

missing_genes = [g for g in genes if g not in gene_table.index]
if missing_genes:
    raise ValueError(f"Selected genes missing from raw source: {missing_genes}")

# Extract the 200 genes in the saved feature order, then transpose.
raw_selected = gene_table.loc[genes]
genomic_features = raw_selected.T
genomic_features.index.name = "patient_id"
genomic_features = genomic_features.reset_index()
genomic_features["patient_id"] = normalize_id(
    genomic_features["patient_id"]
)

if genomic_features["patient_id"].duplicated().any():
    raise ValueError("Duplicate genomic patient IDs after normalization.")

genomic_features = genomic_features[
    genomic_features["patient_id"].isin(needed_ids)
].set_index("patient_id").reindex(sorted(needed_ids))

for gene in genes:
    genomic_features[gene] = pd.to_numeric(
        genomic_features[gene], errors="coerce"
    )

if genomic_features[genes].isna().any().any():
    n_missing = int(genomic_features[genes].isna().sum().sum())
    raise ValueError(
        f"Raw genomic values have {n_missing} missing entries among "
        "the eligible patients. Stop and audit; do not fit imputation "
        "on the full cohort."
    )

genomic_scaler = joblib.load(GENOMIC_SCALER_PATH)

if getattr(genomic_scaler, "n_features_in_", None) != 200:
    raise ValueError("Saved genomic scaler does not expect 200 features.")

if hasattr(genomic_scaler, "feature_names_in_"):
    if list(genomic_scaler.feature_names_in_) != genes:
        raise ValueError(
            "Saved scaler feature order differs from the selected gene list."
        )

genomic_features.loc[:, genes] = genomic_scaler.transform(
    genomic_features[genes]
)

# ------------------------------------------------------------
# 7. Build final and pretraining feature tables by patient ID
# ------------------------------------------------------------
clinical_features = clinical_features.reset_index()
clinical_features["patient_id"] = normalize_id(
    clinical_features["patient_id"]
)

genomic_features = genomic_features.reset_index()
genomic_features["patient_id"] = normalize_id(
    genomic_features["patient_id"]
)

def assemble(ids, include_group=False):
    ids = set(ids)
    ids_df = pd.DataFrame({"patient_id": sorted(ids)})

    out = ids_df.merge(
        clinical_features, on="patient_id",
        how="left", validate="one_to_one"
    ).merge(
        genomic_features, on="patient_id",
        how="left", validate="one_to_one"
    ).merge(
        endpoint, on="patient_id",
        how="left", validate="one_to_one"
    )

    if len(out) != len(ids):
        raise ValueError("Patient count changed during merge.")

    if out["patient_id"].duplicated().any():
        raise ValueError("Duplicate patient IDs after merge.")

    if out.isna().any().any():
        missing_counts = out.isna().sum()
        raise ValueError(
            "Missing values after merge:\n"
            + missing_counts[missing_counts > 0].to_string()
        )

    if include_group:
        group_lookup = master.set_index("patient_id")["cohort_group"]
        out["cohort_group"] = out["patient_id"].map(group_lookup)
        if out["cohort_group"].isna().any():
            raise ValueError("Missing cohort group for pool patient.")

    return out

pool_df = assemble(pool_ids, include_group=True)
train_df = assemble(train_ids)
val_df = assemble(val_ids)
test_df = assemble(test_ids)

# Explicit leakage checks.
assert len(pool_df) == 958
assert len(train_df) == 236
assert len(val_df) == 49
assert len(test_df) == 52

assert set(pool_df["patient_id"]) == pool_ids
assert not (set(pool_df["patient_id"]) & protected_ids)
assert set(train_df["patient_id"]) == train_ids
assert set(val_df["patient_id"]) == val_ids
assert set(test_df["patient_id"]) == test_ids

# Verify feature schema and ordering.
expected_cols = (
    ["patient_id"] + clinical_feature_order + genes
    + ["survival_time", "event"]
)

for name, df in [
    ("pretraining pool", pool_df),
    ("final train", train_df),
    ("final validation", val_df),
    ("final test", test_df),
]:
    actual = [c for c in expected_cols if c in df.columns]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing expected columns: {missing}")

# ------------------------------------------------------------
# 8. Save new outputs separately
# ------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Complete tables: clinical + genomic + RFS
pool_df.to_csv(OUT_DIR / "cg_pretraining_pool_958.csv", index=False)
train_df.to_csv(OUT_DIR / "cg_train_final_236.csv", index=False)
val_df.to_csv(OUT_DIR / "cg_validation_final_49.csv", index=False)
test_df.to_csv(OUT_DIR / "cg_test_final_52.csv", index=False)

# Save separate feature matrices too.
clinical_features[
    clinical_features["patient_id"].isin(pool_ids)
].to_csv(OUT_DIR / "clinical_pretraining_pool.csv", index=False)

genomic_features[
    genomic_features["patient_id"].isin(pool_ids)
].to_csv(OUT_DIR / "genomic_pretraining_pool_200.csv", index=False)

clinical_features[
    clinical_features["patient_id"].isin(train_ids)
].to_csv(OUT_DIR / "clinical_train_final_236.csv", index=False)

clinical_features[
    clinical_features["patient_id"].isin(val_ids)
].to_csv(OUT_DIR / "clinical_validation_final_49.csv", index=False)

clinical_features[
    clinical_features["patient_id"].isin(test_ids)
].to_csv(OUT_DIR / "clinical_test_final_52.csv", index=False)

genomic_features[
    genomic_features["patient_id"].isin(train_ids)
].to_csv(OUT_DIR / "genomic_train_final_236.csv", index=False)

genomic_features[
    genomic_features["patient_id"].isin(val_ids)
].to_csv(OUT_DIR / "genomic_validation_final_49.csv", index=False)

genomic_features[
    genomic_features["patient_id"].isin(test_ids)
].to_csv(OUT_DIR / "genomic_test_final_52.csv", index=False)

# Save the fitted clinical scaler and train-only imputation parameters.
joblib.dump(clinical_scaler, OUT_DIR / "clinical_continuous_scaler_train236.pkl")

with open(OUT_DIR / "clinical_train_only_imputation_parameters.json", "w") as f:
    json.dump(
        {
            "numeric_medians_fit_on_final_train_only": numeric_medians,
            "categorical_modes_fit_on_final_train_only": categorical_modes,
            "continuous_columns_scaled": continuous_cols,
            "clinical_feature_order": clinical_feature_order,
            "selected_genes": genes,
            "train_patients": len(train_ids),
            "validation_patients": len(val_ids),
            "test_patients": len(test_ids),
            "pretraining_pool_patients": len(pool_ids),
        },
        f, indent=2
    )

# Save a concise report.
summary = pd.DataFrame([
    {
        "dataset": "CG pretraining pool",
        "patients": len(pool_df),
        "events": int(pool_df["event"].sum()),
        "censored": int((pool_df["event"] == 0).sum()),
        "clinical_features": len(clinical_feature_order),
        "genomic_features": len(genes),
    },
    {
        "dataset": "Final train",
        "patients": len(train_df),
        "events": int(train_df["event"].sum()),
        "censored": int((train_df["event"] == 0).sum()),
        "clinical_features": len(clinical_feature_order),
        "genomic_features": len(genes),
    },
    {
        "dataset": "Final validation",
        "patients": len(val_df),
        "events": int(val_df["event"].sum()),
        "censored": int((val_df["event"] == 0).sum()),
        "clinical_features": len(clinical_feature_order),
        "genomic_features": len(genes),
    },
    {
        "dataset": "Final test",
        "patients": len(test_df),
        "events": int(test_df["event"].sum()),
        "censored": int((test_df["event"] == 0).sum()),
        "clinical_features": len(clinical_feature_order),
        "genomic_features": len(genes),
    },
])

summary.to_csv(OUT_DIR / "preprocessing_summary.csv", index=False)

print("=" * 70)
print("LEAKAGE-FREE PREPROCESSING COMPLETED")
print("=" * 70)
print("Output directory:", OUT_DIR)
print("\n", summary.to_string(index=False))
print("\nFeature counts:")
print("Clinical:", len(clinical_feature_order))
print("Genomic:", len(genes))
print("Total input features:", len(clinical_feature_order) + len(genes))

print("\nLeakage checks:")
print("Pretraining pool size:", len(pool_df))
print("Validation/test patients in pool:",
      len(set(pool_df["patient_id"]) & protected_ids))
print("Final train IDs match:", set(train_df["patient_id"]) == train_ids)
print("Final validation IDs match:",
      set(val_df["patient_id"]) == val_ids)
print("Final test IDs match:", set(test_df["patient_id"]) == test_ids)

print("\nAll newly generated files are in the new output folder.")
print("Existing clinical, genomic, split, endpoint and pathology files were not modified.")


LEAKAGE-FREE PREPROCESSING COMPLETED
Output directory: /content/gdrive/MyDrive/TCGA_BRCA/clinical_genomic_full_preprocessing_leakage_free_v1/preprocessed_features_v1

             dataset  patients  events  censored  clinical_features  genomic_features
CG pretraining pool       958      59       899                 20               200
        Final train       236      54       182                 20               200
   Final validation        49      12        37                 20               200
         Final test        52      12        40                 20               200

Feature counts:
Clinical: 20
Genomic: 200
Total input features: 220

Leakage checks:
Pretraining pool size: 958
Validation/test patients in pool: 0
Final train IDs match: True
Final validation IDs match: True
Final test IDs match: True

All newly generated files are in the new output folder.
Existing clinical, genomic, split, endpoint and pathology files were not modified.


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/TCGA_BRCA"
OUT = os.path.join(
    BASE,
    "clinical_genomic_full_preprocessing_leakage_free_v1",
    "preprocessed_features_v1"
)
SPLIT_DIR = os.path.join(BASE, "dataset_split_rfs_endpoint_v2")
ENDPOINT_PATH = (
    "/content/gdrive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

FILES = {
    "pool": "cg_pretraining_pool_958.csv",
    "train": "cg_train_final_236.csv",
    "validation": "cg_validation_final_49.csv",
    "test": "cg_test_final_52.csv",
}

def normalize_ids(series):
    return (
        series.astype(str).str.strip().str.upper()
        .str.replace(".", "-", regex=False)
    )

print("=" * 72)
print("FINAL CLINICAL-GENOMIC FILE AUDIT")
print("=" * 72)

# 1. Load the generated datasets
data = {}
for name, filename in FILES.items():
    path = os.path.join(OUT, filename)
    assert os.path.isfile(path), f"Missing file: {path}"
    df = pd.read_csv(path)
    assert "patient_id" in df.columns, f"{filename}: missing patient_id"
    assert {"survival_time", "event"}.issubset(df.columns), (
        f"{filename}: missing survival endpoint columns"
    )
    df["patient_id"] = normalize_ids(df["patient_id"])
    data[name] = df
    print(f"\n{name.upper()}: {df.shape}")
    print("Unique patients:", df["patient_id"].nunique())
    print("Events:", int(df["event"].sum()))
    print("Censored:", int((df["event"] == 0).sum()))
    print("Duplicate patient IDs:", int(df["patient_id"].duplicated().sum()))

# 2. Check expected sizes and unique IDs
expected_sizes = {"pool": 958, "train": 236, "validation": 49, "test": 52}
for name, expected in expected_sizes.items():
    assert len(data[name]) == expected, (
        f"{name}: expected {expected} rows, got {len(data[name])}"
    )
    assert data[name]["patient_id"].nunique() == expected, (
        f"{name}: duplicate patient IDs found"
    )

# 3. Load the protected final split IDs
split_files = {
    "train": "train_rfs_patient_ids.csv",
    "validation": "validation_rfs_patient_ids.csv",
    "test": "test_rfs_patient_ids.csv",
}
split_ids = {}
for name, filename in split_files.items():
    split_df = pd.read_csv(os.path.join(SPLIT_DIR, filename))
    assert "patient_id" in split_df.columns, (
        f"{filename}: expected a patient_id column"
    )
    split_ids[name] = set(normalize_ids(split_df["patient_id"]))

for name in ("train", "validation", "test"):
    actual = set(data[name]["patient_id"])
    print(f"\n{name.upper()} IDs exactly match protected split:",
          actual == split_ids[name])
    assert actual == split_ids[name], f"{name} IDs do not match the split!"

pool_ids = set(data["pool"]["patient_id"])
print("\nValidation IDs in pretraining pool:",
      len(pool_ids & split_ids["validation"]))
print("Test IDs in pretraining pool:",
      len(pool_ids & split_ids["test"]))
assert not (pool_ids & split_ids["validation"])
assert not (pool_ids & split_ids["test"])

extra_ids = pool_ids - split_ids["train"]
print("Additional eligible pretraining patients:", len(extra_ids))
assert len(pool_ids & split_ids["train"]) == 236
assert len(extra_ids) == 722
assert pool_ids == split_ids["train"] | extra_ids

# 4. Check feature schema and missing/non-finite values
feature_cols = [
    c for c in data["pool"].columns
    if c not in {
        "patient_id",
        "survival_time",
        "event",
        "cohort_group"
    }
]
assert len(feature_cols) == 220
assert "cohort_group" in data["pool"].columns
assert "cohort_group" not in feature_cols

print("Clinical + genomic input features:", len(feature_cols))
print("Metadata excluded from model input: cohort_group")

print("\nInput feature count:", len(feature_cols))
assert len(feature_cols) == 220, (
    f"Expected 220 clinical+genomic features, got {len(feature_cols)}"
)

pool_feature_set = set(feature_cols)
for name, df in data.items():
    cols = [
    c for c in df.columns
    if c not in {
        "patient_id",
        "survival_time",
        "event",
        "cohort_group"
    }
]
    assert set(cols) == pool_feature_set, (
        f"{name}: feature columns differ from pretraining pool"
    )
    assert df[feature_cols].notna().all().all(), (
        f"{name}: missing feature values detected"
    )
    assert np.isfinite(df[feature_cols].to_numpy(dtype=float)).all(), (
        f"{name}: non-finite feature values detected"
    )
    assert df["survival_time"].notna().all()
    assert (df["survival_time"] > 0).all()
    assert df["event"].isin([0, 1]).all()

# 5. Compare labels against the audited endpoint file
endpoint = pd.read_csv(ENDPOINT_PATH)
endpoint["patient_id"] = normalize_ids(endpoint["patient_id"])
endpoint = endpoint[["patient_id", "survival_time", "event"]]
assert not endpoint["patient_id"].duplicated().any()

endpoint = endpoint.set_index("patient_id")
for name, df in data.items():
    check = df.set_index("patient_id").join(
        endpoint, how="left", rsuffix="_source"
    )
    assert check["survival_time_source"].notna().all(), (
        f"{name}: some patients missing from audited endpoint"
    )
    assert np.allclose(
        check["survival_time"], check["survival_time_source"]
    ), f"{name}: survival times differ from audited endpoint"
    assert np.array_equal(
        check["event"].to_numpy(),
        check["event_source"].to_numpy()
    ), f"{name}: event labels differ from audited endpoint"

print("\n" + "=" * 72)
print("ALL CHECKS PASSED")
print("Files are consistent with the protected splits and audited endpoints.")
print("These four CSV files contain clinical + genomic features only.")
print("No validation/test patients were used in the pretraining pool.")
print("No files were modified.")
print("=" * 72)


FINAL CLINICAL-GENOMIC FILE AUDIT

POOL: (958, 224)
Unique patients: 958
Events: 59
Censored: 899
Duplicate patient IDs: 0

TRAIN: (236, 223)
Unique patients: 236
Events: 54
Censored: 182
Duplicate patient IDs: 0

VALIDATION: (49, 223)
Unique patients: 49
Events: 12
Censored: 37
Duplicate patient IDs: 0

TEST: (52, 223)
Unique patients: 52
Events: 12
Censored: 40
Duplicate patient IDs: 0

TRAIN IDs exactly match protected split: True

VALIDATION IDs exactly match protected split: True

TEST IDs exactly match protected split: True

Validation IDs in pretraining pool: 0
Test IDs in pretraining pool: 0
Additional eligible pretraining patients: 722
Clinical + genomic input features: 220
Metadata excluded from model input: cohort_group

Input feature count: 220

ALL CHECKS PASSED
Files are consistent with the protected splits and audited endpoints.
These four CSV files contain clinical + genomic features only.
No validation/test patients were used in the pretraining pool.
No files were modi

In [ ]:

import os
import pandas as pd

OUT = (
    "/content/gdrive/MyDrive/TCGA_BRCA/"
    "clinical_genomic_full_preprocessing_leakage_free_v1/"
    "preprocessed_features_v1"
)

pool = pd.read_csv(os.path.join(OUT, "cg_pretraining_pool_958.csv"))
train = pd.read_csv(os.path.join(OUT, "cg_train_final_236.csv"))

print("Columns in pool but not in train:")
print(sorted(set(pool.columns) - set(train.columns)))

print("\nColumns in train but not in pool:")
print(sorted(set(train.columns) - set(pool.columns)))

print("\nPool columns that are not numeric:")
print(pool.select_dtypes(exclude="number").columns.tolist())

print("\nLast 10 pool columns:")
print(pool.columns[-10:].tolist())

print("\nLast 10 train columns:")
print(train.columns[-10:].tolist())


Columns in pool but not in train:
['cohort_group']

Columns in train but not in pool:
[]

Pool columns that are not numeric:
['patient_id', 'cohort_group']

Last 10 pool columns:
['VWA2', 'C6orf141', 'RPS28', 'MUC2', 'S100A7', 'GP2', 'L1CAM', 'survival_time', 'event', 'cohort_group']

Last 10 train columns:
['C5orf46', 'VWA2', 'C6orf141', 'RPS28', 'MUC2', 'S100A7', 'GP2', 'L1CAM', 'survival_time', 'event']


In [ ]:

import os
import joblib
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/TCGA_BRCA"
OUT = os.path.join(
    BASE,
    "clinical_genomic_full_preprocessing_leakage_free_v1",
    "preprocessed_features_v1"
)
GENOMIC_DIR = os.path.join(
    BASE, "rfs_feature_selection_leakage_free_endpoint_v2"
)

pool = pd.read_csv(os.path.join(OUT, "cg_pretraining_pool_958.csv"))
train = pd.read_csv(os.path.join(OUT, "cg_train_final_236.csv"))

pool["patient_id"] = (
    pool["patient_id"].astype(str).str.strip().str.upper()
)
train["patient_id"] = (
    train["patient_id"].astype(str).str.strip().str.upper()
)

# 1. Verify selected genes
gene_file = os.path.join(GENOMIC_DIR, "rfs_selected_genes_200.csv")
genes_df = pd.read_csv(gene_file)
genes = genes_df["rfs_selected_genes"].astype(str).tolist()

assert len(genes) == 200, f"Expected 200 genes, got {len(genes)}"
assert len(set(genes)) == 200, "Duplicate genes in selected-gene list"

missing_genes = [g for g in genes if g not in pool.columns]
assert not missing_genes, f"Missing selected genes: {missing_genes}"

print("=" * 70)
print("1. SELECTED GENE CHECK")
print("Selected genes:", len(genes))
print("All selected genes present:", len(missing_genes) == 0)

# 2. Verify all 236 training rows match their copies in the pool
feature_cols = [
    c for c in train.columns
    if c not in {"patient_id", "survival_time", "event"}
]
assert len(feature_cols) == 220
assert set(feature_cols) <= set(pool.columns)

pool_train = (
    pool[pool["patient_id"].isin(set(train["patient_id"]))]
    .set_index("patient_id")
    .sort_index()
)
train_indexed = train.set_index("patient_id").sort_index()

assert pool_train.index.equals(train_indexed.index)
assert np.allclose(
    pool_train[feature_cols].to_numpy(dtype=float),
    train_indexed[feature_cols].to_numpy(dtype=float),
    rtol=1e-6,
    atol=1e-7
), "Training features differ between pool and train CSV!"

print("\n2. TRAINING FEATURE CONSISTENCY")
print("All 236 training patients match the pool:", True)
print("Clinical + genomic feature count:", len(feature_cols))

# 3. Check the saved genomic scaler
scaler_path = os.path.join(
    GENOMIC_DIR, "genomic_standard_scaler_200.pkl"
)
scaler = joblib.load(scaler_path)

assert scaler.n_features_in_ == 200
print("\n3. SAVED GENOMIC SCALER")
print("Scaler type:", type(scaler).__name__)
print("Expected input features:", scaler.n_features_in_)

X_train_genes = train_indexed[genes].to_numpy(dtype=float)
assert np.isfinite(X_train_genes).all()

train_means = X_train_genes.mean(axis=0)
train_stds = X_train_genes.std(axis=0, ddof=0)

print("Largest absolute training gene mean:", float(np.abs(train_means).max()))
print("Smallest training gene SD:", float(train_stds.min()))
print("Largest training gene SD:", float(train_stds.max()))

# 4. Check clinical feature values for unexpected fractions
clinical_cols = [c for c in feature_cols if c not in genes]
assert len(clinical_cols) == 20

clinical_train = train_indexed[clinical_cols].apply(
    pd.to_numeric, errors="raise"
)

binary_cols = [
    c for c in clinical_cols
    if c.startswith("histological_type_")
    or c.startswith("PAM50_")
    or c.startswith("race_")
    or c.startswith("ethnicity_")
    or c == "radiation_therapy"
]

fractional_binary = {}
for col in binary_cols:
    vals = clinical_train[col].dropna().to_numpy(dtype=float)
    unusual = sorted(set(vals) - {0.0, 1.0})
    if unusual:
        fractional_binary[col] = unusual[:10]

print("\n4. CLINICAL FEATURE CHECK")
print("Clinical feature count:", len(clinical_cols))
print("Binary/dummy columns checked:", len(binary_cols))
print("Binary columns with values other than 0/1:", fractional_binary)

assert not fractional_binary, (
    "Unexpected binary feature values found. Inspect the output above."
)

print("\n" + "=" * 70)
print("FEATURE VERIFICATION PASSED")
print("No files were modified.")
print("=" * 70)


1. SELECTED GENE CHECK
Selected genes: 200
All selected genes present: True

2. TRAINING FEATURE CONSISTENCY
All 236 training patients match the pool: True
Clinical + genomic feature count: 220

3. SAVED GENOMIC SCALER
Scaler type: StandardScaler
Expected input features: 200
Largest absolute training gene mean: 8.279629336187608e-16
Smallest training gene SD: 0.9999999999999998
Largest training gene SD: 1.0000000000000002

4. CLINICAL FEATURE CHECK
Clinical feature count: 20
Binary/dummy columns checked: 13
Binary columns with values other than 0/1: {}

FEATURE VERIFICATION PASSED
No files were modified.


In [ ]:

import os
import pandas as pd

BASE = "/content/gdrive/MyDrive/TCGA_BRCA"
SPLIT_DIR = os.path.join(BASE, "dataset_split_rfs_endpoint_v2")
PATHOLOGY_DIR = os.path.join(
    BASE, "multimodal_rfs_dataset_leakage_free", "pathology_512"
)
MANIFEST_PATH = os.path.join(
    PATHOLOGY_DIR, "pathology_512_manifest.csv"
)

def norm_ids(s):
    return (
        s.astype(str).str.strip().str.upper()
        .str.replace(".", "-", regex=False)
    )

# 1. Load final split IDs
split_files = {
    "train": "train_rfs_patient_ids.csv",
    "validation": "validation_rfs_patient_ids.csv",
    "test": "test_rfs_patient_ids.csv",
}
splits = {}
for name, filename in split_files.items():
    df = pd.read_csv(os.path.join(SPLIT_DIR, filename))
    splits[name] = set(norm_ids(df["patient_id"]))

# 2. Load pathology manifest
assert os.path.isfile(MANIFEST_PATH), (
    f"Pathology manifest not found: {MANIFEST_PATH}"
)

manifest = pd.read_csv(MANIFEST_PATH)
print("Manifest columns:", manifest.columns.tolist())
assert {"patient_id", "file_uuid"}.issubset(manifest.columns), (
    "Expected patient_id and file_uuid columns."
)

manifest["patient_id"] = norm_ids(manifest["patient_id"])
pathology_ids = set(manifest["patient_id"].dropna())

print("\n" + "=" * 70)
print("FINAL PATHOLOGY ELIGIBILITY AUDIT")
print("=" * 70)
print("Manifest rows:", len(manifest))
print("Unique pathology patients:", len(pathology_ids))
print("Duplicate patient rows:", int(manifest["patient_id"].duplicated().sum()))

# 3. Check patient coverage in each final split
for name, ids in splits.items():
    matched = ids & pathology_ids
    missing = ids - pathology_ids

    print(f"\n{name.upper()}:")
    print("Patients in split:", len(ids))
    print("Patients with pathology entry:", len(matched))
    print("Patients without pathology entry:", len(missing))
    print("Missing IDs (first 15):", sorted(missing)[:15])

# 4. Check whether manifest UUIDs have matching .npy files
npy_files = {
    f for f in os.listdir(PATHOLOGY_DIR)
    if f.endswith(".npy")
}
npy_stems = {os.path.splitext(f)[0] for f in npy_files}

manifest_uuids = set(manifest["file_uuid"].dropna().astype(str))
matched_uuids = manifest_uuids & npy_stems

print("\nPATHOLOGY FILE CHECK")
print("Unique manifest UUIDs:", len(manifest_uuids))
print("Matching .npy files:", len(matched_uuids))
print("UUIDs without matching .npy:", len(manifest_uuids - npy_stems))
print("Missing UUID examples:", sorted(manifest_uuids - npy_stems)[:10])

print("\nAudit completed. No files were modified.")


Manifest columns: ['patient_id', 'file_uuid', 'original_num_patches', 'selected_num_patches', 'padded_patches', 'feature_dim']

FINAL PATHOLOGY ELIGIBILITY AUDIT
Manifest rows: 357
Unique pathology patients: 357
Duplicate patient rows: 0

TRAIN:
Patients in split: 236
Patients with pathology entry: 236
Patients without pathology entry: 0
Missing IDs (first 15): []

VALIDATION:
Patients in split: 49
Patients with pathology entry: 49
Patients without pathology entry: 0
Missing IDs (first 15): []

TEST:
Patients in split: 52
Patients with pathology entry: 52
Patients without pathology entry: 0
Missing IDs (first 15): []

PATHOLOGY FILE CHECK
Unique manifest UUIDs: 357
Matching .npy files: 0
UUIDs without matching .npy: 357
Missing UUID examples: ['00ab866f-a0a3-44c0-ad14-c9dc833239a7', '01c79d06-a4a7-47a7-bb8b-b98ce6652c99', '02732da3-5144-4680-b6ee-cb8563125493', '0279f555-9738-41e0-b89f-ded25829adc0', '032fc1ed-c515-4b26-a229-f67c1eb6fdc3', '035f4d0d-78f8-4df0-94a2-112922f584a8', '03852

In [ ]:

import os

BASES = [
    "/content/gdrive/MyDrive/TCGA_BRCA",
    "/content/drive/MyDrive/TCGA_BRCA",
]

print("Checking Google Drive mount...")
for base in BASES:
    print(f"{base}: exists = {os.path.isdir(base)}")

# Search likely TCGA_BRCA folders for the manifest
found = []
for base in BASES:
    if not os.path.isdir(base):
        continue

    for root, dirs, files in os.walk(base):
        # Avoid searching unrelated hidden/system folders
        dirs[:] = [d for d in dirs if not d.startswith(".")]

        for filename in files:
            low = filename.lower()
            if (
                filename.lower().endswith(".csv")
                and "manifest" in low
                and ("pathology" in low or "512" in low)
            ):
                found.append(os.path.join(root, filename))

print("\nPathology manifest candidates:")
if found:
    for path in found:
        print(path)
else:
    print("No matching manifest found in the searched folders.")

# Also locate folders containing .npy pathology features
print("\nFolders containing .npy files:")
npy_folders = set()
for base in BASES:
    if not os.path.isdir(base):
        continue
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        if any(f.endswith(".npy") for f in files):
            npy_folders.add(root)

for path in sorted(npy_folders):
    print(path)


Checking Google Drive mount...
/content/gdrive/MyDrive/TCGA_BRCA: exists = True
/content/drive/MyDrive/TCGA_BRCA: exists = True

Pathology manifest candidates:
/content/gdrive/MyDrive/TCGA_BRCA/multimodal_rfs_dataset_leakage_free/pathology_512/pathology_512_manifest.csv

Folders containing .npy files:
/content/gdrive/MyDrive/TCGA_BRCA/Features
/content/gdrive/MyDrive/TCGA_BRCA/multimodal_rfs_dataset_leakage_free/pathology_512


In [ ]:

import os
import glob
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/TCGA_BRCA"

MANIFEST_PATH = os.path.join(
    BASE,
    "multimodal_rfs_dataset_leakage_free",
    "pathology_512",
    "pathology_512_manifest.csv"
)

FOLDERS = {
    "pathology_512": os.path.join(
        BASE, "multimodal_rfs_dataset_leakage_free", "pathology_512"
    ),
    "Features": os.path.join(BASE, "Features")
}

manifest = pd.read_csv(MANIFEST_PATH)

print("=" * 65)
print("1. MANIFEST SAMPLE")
print("=" * 65)
print(manifest[["patient_id", "file_uuid"]].head(5).to_string(index=False))

for folder_name, folder in FOLDERS.items():
    print("\n" + "=" * 65)
    print(f"2. FOLDER: {folder_name}")
    print("Path exists:", os.path.isdir(folder))

    if not os.path.isdir(folder):
        continue

    # Search recursively, without changing any files.
    files = glob.glob(os.path.join(folder, "**", "*.npy"), recursive=True)

    print("Number of .npy files:", len(files))
    print("First 10 filenames:")
    for path in files[:10]:
        print(" ", os.path.relpath(path, folder))

    if files:
        sample_path = files[0]
        print("\nInspecting one sample:")
        print("Filename:", os.path.basename(sample_path))

        try:
            arr = np.load(sample_path, mmap_mode="r", allow_pickle=False)
            print("Shape:", arr.shape)
            print("Data type:", arr.dtype)
            print("Dimensions:", arr.ndim)
        except Exception as e:
            print("Could not inspect sample:", repr(e))

print("\n" + "=" * 65)
print("3. MANIFEST UUID EXAMPLES")
print("=" * 65)
print(manifest["file_uuid"].astype(str).head(10).tolist())


1. MANIFEST SAMPLE
  patient_id                            file_uuid
TCGA-5L-AAT1 4eec69ca-381b-4c17-b3e9-49492d71560e
TCGA-A1-A0SB cea82b7d-135a-49d5-b4f6-3fb0215f7188
TCGA-A1-A0SF f273a858-1cfa-4983-ace4-b1e876d64473
TCGA-A1-A0SI a951559a-e6c0-419b-b09f-43e6cb6ee3ea
TCGA-A1-A0SJ 9a3a3830-9674-4f9b-985f-e50062e78807

2. FOLDER: pathology_512
Path exists: True
Number of .npy files: 714
First 10 filenames:
  TCGA-5L-AAT1.npy
  TCGA-5L-AAT1_mask.npy
  TCGA-A1-A0SB.npy
  TCGA-A1-A0SB_mask.npy
  TCGA-A1-A0SF.npy
  TCGA-A1-A0SF_mask.npy
  TCGA-A1-A0SI.npy
  TCGA-A1-A0SI_mask.npy
  TCGA-A1-A0SJ.npy
  TCGA-A1-A0SJ_mask.npy

Inspecting one sample:
Filename: TCGA-5L-AAT1.npy
Shape: (512, 512)
Data type: float32
Dimensions: 2

2. FOLDER: Features
Path exists: True
Number of .npy files: 372
First 10 filenames:
  a1b58cd0-559e-467d-a193-96602cd3306f.npy
  997dbd8f-9f16-4fc2-8627-d29ee0d74ebe.npy
  9132985f-ecd8-4a67-ba7b-63902801fba6.npy
  ce77af22-10b7-4480-9c93-015970268455.npy
  fcc84daa-5e54-4

In [ ]:

import os
import glob
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/TCGA_BRCA"

PATHOLOGY_DIR = os.path.join(
    BASE, "multimodal_rfs_dataset_leakage_free", "pathology_512"
)
FEATURES_DIR = os.path.join(BASE, "Features")
MANIFEST_PATH = os.path.join(PATHOLOGY_DIR, "pathology_512_manifest.csv")

manifest = pd.read_csv(MANIFEST_PATH)
manifest["patient_id"] = manifest["patient_id"].astype(str).str.strip()
manifest["file_uuid"] = manifest["file_uuid"].astype(str).str.strip()

print("=" * 70)
print("1. UUID FEATURE FILE COVERAGE")
print("=" * 70)

uuid_files = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(os.path.join(FEATURES_DIR, "*.npy"))
}

manifest_uuids = set(manifest["file_uuid"])
matched_uuids = manifest_uuids.intersection(uuid_files)
missing_uuids = manifest_uuids - set(uuid_files)

print("Manifest rows:", len(manifest))
print("Unique manifest UUIDs:", manifest["file_uuid"].nunique())
print("UUID .npy files in Features/:", len(uuid_files))
print("Manifest UUIDs with files:", len(matched_uuids))
print("Manifest UUIDs missing files:", len(missing_uuids))
print("Missing UUID examples:", list(sorted(missing_uuids))[:10])

print("\n" + "=" * 70)
print("2. FEATURE SHAPES VS MANIFEST")
print("=" * 70)

shape_results = []
for _, row in manifest.iterrows():
    uid = row["file_uuid"]
    path = uuid_files.get(uid)

    if path is None:
        shape_results.append({
            "patient_id": row["patient_id"],
            "file_uuid": uid,
            "status": "MISSING_FILE",
            "actual_shape": None,
            "manifest_selected_patches": row.get("selected_num_patches"),
            "manifest_feature_dim": row.get("feature_dim")
        })
        continue

    try:
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        actual_shape = tuple(arr.shape)
        shape_results.append({
            "patient_id": row["patient_id"],
            "file_uuid": uid,
            "status": "OK",
            "actual_shape": str(actual_shape),
            "manifest_selected_patches": row.get("selected_num_patches"),
            "manifest_feature_dim": row.get("feature_dim")
        })
    except Exception as e:
        shape_results.append({
            "patient_id": row["patient_id"],
            "file_uuid": uid,
            "status": "LOAD_ERROR: " + repr(e),
            "actual_shape": None,
            "manifest_selected_patches": row.get("selected_num_patches"),
            "manifest_feature_dim": row.get("feature_dim")
        })

shape_df = pd.DataFrame(shape_results)
print("File status counts:")
print(shape_df["status"].value_counts(dropna=False).to_string())

print("\nFirst 10 shape records:")
print(shape_df.head(10).to_string(index=False))

print("\nObserved feature shapes (up to 15):")
print(shape_df.loc[shape_df["actual_shape"].notna(), "actual_shape"]
      .value_counts().head(15).to_string())

print("\nManifest selected_num_patches values:")
print(manifest["selected_num_patches"].value_counts(dropna=False).head(10).to_string())

print("\nManifest feature_dim values:")
print(manifest["feature_dim"].value_counts(dropna=False).head(10).to_string())

print("\n" + "=" * 70)
print("3. PATIENT-ID ARRAYS AND MASKS")
print("=" * 70)

patient_files = set()
mask_files = set()

for p in glob.glob(os.path.join(PATHOLOGY_DIR, "*.npy")):
    stem = os.path.splitext(os.path.basename(p))[0]
    if stem.endswith("_mask"):
        mask_files.add(stem[:-5])
    else:
        patient_files.add(stem)

manifest_patients = set(manifest["patient_id"])

print("Patient-level arrays:", len(patient_files))
print("Mask arrays:", len(mask_files))
print("Manifest patients with patient-level arrays:",
      len(manifest_patients & patient_files))
print("Manifest patients missing patient-level arrays:",
      len(manifest_patients - patient_files))
print("Manifest patients with masks:",
      len(manifest_patients & mask_files))
print("Manifest patients missing masks:",
      len(manifest_patients - mask_files))

print("\nPatient array examples:")
for pid in sorted(manifest_patients & patient_files)[:3]:
    p = os.path.join(PATHOLOGY_DIR, pid + ".npy")
    try:
        arr = np.load(p, mmap_mode="r", allow_pickle=False)
        print(pid, "shape =", arr.shape, "dtype =", arr.dtype)
    except Exception as e:
        print(pid, "ERROR:", repr(e))

print("\nAudit complete. No files were modified.")


1. UUID FEATURE FILE COVERAGE
Manifest rows: 357
Unique manifest UUIDs: 357
UUID .npy files in Features/: 372
Manifest UUIDs with files: 357
Manifest UUIDs missing files: 0
Missing UUID examples: []

2. FEATURE SHAPES VS MANIFEST
File status counts:
status
LOAD_ERROR: OSError(107, 'Transport endpoint is not connected')    353
OK                                                                   4

First 10 shape records:
  patient_id                            file_uuid                                                          status actual_shape  manifest_selected_patches  manifest_feature_dim
TCGA-5L-AAT1 4eec69ca-381b-4c17-b3e9-49492d71560e                                                              OK  (1777, 512)                        512                   512
TCGA-A1-A0SB cea82b7d-135a-49d5-b4f6-3fb0215f7188                                                              OK  (4512, 512)                        512                   512
TCGA-A1-A0SF f273a858-1cfa-4983-ace4-b1e876d6447

In [ ]:

from google.colab import drive
import os
import numpy as np

drive.mount("/content/gdrive", force_remount=True)

BASE = "/content/gdrive/MyDrive/TCGA_BRCA"
PATHOLOGY_DIR = os.path.join(
    BASE, "multimodal_rfs_dataset_leakage_free", "pathology_512"
)
FEATURES_DIR = os.path.join(BASE, "Features")

print("Pathology folder exists:", os.path.isdir(PATHOLOGY_DIR))
print("Features folder exists:", os.path.isdir(FEATURES_DIR))

patient_files = [
    f for f in os.listdir(PATHOLOGY_DIR)
    if f.endswith(".npy") and not f.endswith("_mask.npy")
]
mask_files = [
    f for f in os.listdir(PATHOLOGY_DIR)
    if f.endswith("_mask.npy")
]

print("Patient-level arrays:", len(patient_files))
print("Mask arrays:", len(mask_files))
print("Example patient arrays:", patient_files[:5])

# Inspect only one small patient-level array.
if patient_files:
    sample_path = os.path.join(PATHOLOGY_DIR, patient_files[0])
    arr = np.load(sample_path, allow_pickle=False)
    print("Sample:", patient_files[0])
    print("Shape:", arr.shape)
    print("Data type:", arr.dtype)


Mounted at /content/gdrive
Pathology folder exists: True
Features folder exists: True
Patient-level arrays: 357
Mask arrays: 357
Example patient arrays: ['TCGA-5L-AAT1.npy', 'TCGA-A1-A0SB.npy', 'TCGA-A1-A0SF.npy', 'TCGA-A1-A0SI.npy', 'TCGA-A1-A0SJ.npy']
Sample: TCGA-5L-AAT1.npy
Shape: (512, 512)
Data type: float32


##Imports and paths

In [ ]:
# ============================================================
# CLINICAL-GENOMIC PRETRAINING — LEAKAGE-FREE V1
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BASE_DIR = "/content/gdrive/MyDrive/TCGA_BRCA"

PREPROCESS_DIR = os.path.join(
    BASE_DIR,
    "clinical_genomic_full_preprocessing_leakage_free_v1",
    "preprocessed_features_v1"
)

SPLIT_DIR = os.path.join(
    BASE_DIR,
    "dataset_split_rfs_endpoint_v2"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "clinical_genomic_pretraining_leakage_free_v1"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

POOL_PATH = os.path.join(
    PREPROCESS_DIR, "cg_pretraining_pool_958.csv"
)

TRAIN_PATH = os.path.join(
    PREPROCESS_DIR, "cg_train_final_236.csv"
)

VAL_PATH = os.path.join(
    PREPROCESS_DIR, "cg_validation_final_49.csv"
)

TEST_PATH = os.path.join(
    PREPROCESS_DIR, "cg_test_final_52.csv"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for path in [POOL_PATH, TRAIN_PATH, VAL_PATH, TEST_PATH]:
    assert os.path.isfile(path), f"Missing file: {path}"

print("\nAll four prepared datasets exist.")
print("Output folder:", OUTPUT_DIR)

Device: cpu

All four prepared datasets exist.
Output folder: /content/gdrive/MyDrive/TCGA_BRCA/clinical_genomic_pretraining_leakage_free_v1


##Load and audit the four datasets

In [ ]:
# ============================================================
# LOAD PREPROCESSED DATASETS
# ============================================================

pool = pd.read_csv(POOL_PATH)
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VAL_PATH)
test = pd.read_csv(TEST_PATH)

datasets = {
    "Pretraining pool": pool,
    "Final training": train,
    "Final validation": validation,
    "Final test": test
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Unique patients:", df["patient_id"].nunique())
    print("Events:", int(df["event"].sum()))
    print("Censored:", int((df["event"] == 0).sum()))

    assert df["patient_id"].is_unique, (
        f"Duplicate patient IDs in {name}"
    )

    assert df["event"].isin([0, 1]).all(), (
        f"Unexpected event values in {name}"
    )

    assert (df["survival_time"] > 0).all(), (
        f"Non-positive survival times in {name}"
    )

print("\nFirst columns in the pretraining pool:")
print(pool.columns.tolist()[:30])


Pretraining pool
Shape: (958, 224)
Unique patients: 958
Events: 59
Censored: 899

Final training
Shape: (236, 223)
Unique patients: 236
Events: 54
Censored: 182

Final validation
Shape: (49, 223)
Unique patients: 49
Events: 12
Censored: 37

Final test
Shape: (52, 223)
Unique patients: 52
Events: 12
Censored: 40

First columns in the pretraining pool:
['patient_id', 'years_to_birth', 'Tumor_purity', 'pathologic_stage', 'pathology_T_stage', 'pathology_N_stage', 'pathology_M_stage', 'number_of_lymph_nodes', 'radiation_therapy', 'histological_type_infiltratinglobularcarcinoma', 'histological_type_medullarycarcinoma', 'histological_type_metaplasticcarcinoma', 'histological_type_mixedhistology(pleasespecify)', 'histological_type_mucinouscarcinoma', 'histological_type_other,specify', 'PAM50_Her2', 'PAM50_LumA', 'PAM50_LumB', 'race_blackorafricanamerican', 'race_white', 'ethnicity_nothispanicorlatino', 'PSCA', 'ZNF385B', 'MYBPC1', 'CALML5', 'PRAME', 'FLT3', 'GREB1', 'TSPAN8', 'IL6ST']


##Verify that the protected validation and test patients are excluded

In [ ]:
# ============================================================
# VERIFY PROTECTED SPLITS AND PRETRAINING POOL
# ============================================================

def load_ids(filename):
    path = os.path.join(SPLIT_DIR, filename)
    assert os.path.isfile(path), f"Missing split file: {path}"

    df = pd.read_csv(path)

    assert "patient_id" in df.columns, (
        f"'patient_id' column missing in {path}"
    )

    return set(df["patient_id"].astype(str).str.strip())


expected_train = load_ids("train_rfs_patient_ids.csv")
expected_val = load_ids("validation_rfs_patient_ids.csv")
expected_test = load_ids("test_rfs_patient_ids.csv")

pool_ids = set(pool["patient_id"].astype(str).str.strip())
train_ids = set(train["patient_id"].astype(str).str.strip())
val_ids = set(validation["patient_id"].astype(str).str.strip())
test_ids = set(test["patient_id"].astype(str).str.strip())

assert train_ids == expected_train, "Training IDs do not match the protected split."
assert val_ids == expected_val, "Validation IDs do not match the protected split."
assert test_ids == expected_test, "Test IDs do not match the protected split."

assert not (pool_ids & val_ids), (
    "Leakage detected: validation patients appear in the pretraining pool."
)

assert not (pool_ids & test_ids), (
    "Leakage detected: test patients appear in the pretraining pool."
)

assert train_ids.issubset(pool_ids), (
    "The pretraining pool is missing some final training patients."
)

print("Protected split checks passed.")
print("Final training patients:", len(train_ids))
print("Final validation patients:", len(val_ids))
print("Final test patients:", len(test_ids))
print("Additional pretraining patients:", len(pool_ids - train_ids))
print("Validation patients in pool:", len(pool_ids & val_ids))
print("Test patients in pool:", len(pool_ids & test_ids))

Protected split checks passed.
Final training patients: 236
Final validation patients: 49
Final test patients: 52
Additional pretraining patients: 722
Validation patients in pool: 0
Test patients in pool: 0


##Verify the exact model features

In [ ]:
# ============================================================
# CELL 4 — DEFINE AND VERIFY MODEL FEATURES
# ============================================================

CLINICAL_BASE = [
    "years_to_birth",
    "Tumor_purity",
    "pathologic_stage",
    "pathology_T_stage",
    "pathology_N_stage",
    "pathology_M_stage",
    "number_of_lymph_nodes",
    "radiation_therapy",
]

CLINICAL_PREFIXES = (
    "histological_type",
    "PAM50",
    "race",
    "ethnicity",
)

clinical_columns = CLINICAL_BASE + [
    c for c in pool.columns
    if c.startswith(CLINICAL_PREFIXES)
]

METADATA_COLUMNS = {
    "patient_id",
    "event",
    "survival_time",
    "cohort_group",
    "split",
    "file_uuid",
    "pathology_path",
}

genomic_columns = [
    c for c in pool.columns
    if c not in clinical_columns
    and c not in METADATA_COLUMNS
]

# Verify expected feature counts
assert len(clinical_columns) == 20, (
    f"Expected 20 clinical features; found {len(clinical_columns)}"
)

assert len(genomic_columns) == 200, (
    f"Expected 200 genomic features; found {len(genomic_columns)}"
)

# Ensure feature columns are available and identically ordered
for name, df in datasets.items():
    assert all(c in df.columns for c in clinical_columns), (
        f"Missing clinical features in {name}"
    )
    assert all(c in df.columns for c in genomic_columns), (
        f"Missing genomic features in {name}"
    )

    X = df[clinical_columns + genomic_columns].to_numpy(dtype=np.float64)

    assert np.isfinite(X).all(), (
        f"Non-finite model inputs in {name}"
    )

    assert "event" not in clinical_columns + genomic_columns
    assert "survival_time" not in clinical_columns + genomic_columns
    assert "cohort_group" not in clinical_columns + genomic_columns

print("Clinical features:", len(clinical_columns))
print("Genomic features:", len(genomic_columns))
print("Total input features:", len(clinical_columns) + len(genomic_columns))
print("\nFeature audit passed.")
print("\nClinical columns:")
print(clinical_columns)
print("\nFirst 20 genomic columns:")
print(genomic_columns[:20])

Clinical features: 20
Genomic features: 200
Total input features: 220

Feature audit passed.

Clinical columns:
['years_to_birth', 'Tumor_purity', 'pathologic_stage', 'pathology_T_stage', 'pathology_N_stage', 'pathology_M_stage', 'number_of_lymph_nodes', 'radiation_therapy', 'histological_type_infiltratinglobularcarcinoma', 'histological_type_medullarycarcinoma', 'histological_type_metaplasticcarcinoma', 'histological_type_mixedhistology(pleasespecify)', 'histological_type_mucinouscarcinoma', 'histological_type_other,specify', 'PAM50_Her2', 'PAM50_LumA', 'PAM50_LumB', 'race_blackorafricanamerican', 'race_white', 'ethnicity_nothispanicorlatino']

First 20 genomic columns:
['PSCA', 'ZNF385B', 'MYBPC1', 'CALML5', 'PRAME', 'FLT3', 'GREB1', 'TSPAN8', 'IL6ST', 'TUBA3E', 'RP1', 'DUSP9', 'S100P', 'LRRN1', 'FAM83A', 'CACNG4', 'LOC100128977', 'C1orf173', 'LOC100130148', 'HOTAIR']


##Define the clinical–genomic Transformer

In [ ]:
# ============================================================
# CELL 5 — CLINICAL-GENOMIC TRANSFORMER
# ============================================================

class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=128, dropout=0.3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.encoder(x).unsqueeze(1)


class GenomicEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=128, dropout=0.3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x):
        return self.encoder(x).unsqueeze(1)


class ClinicalGenomicFusion(nn.Module):
    def __init__(
        self,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        dropout=0.3,
    ):
        super().__init__()

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, clinical_token, genomic_token):
        x = torch.cat(
            [clinical_token, genomic_token],
            dim=1,
        )

        x = self.transformer(x)
        x = self.norm(x)

        return x.mean(dim=1)


class SurvivalPredictionHead(nn.Module):
    def __init__(self, embed_dim=128, dropout=0.3):
        super().__init__()

        self.head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.head(x)


class ClinicalGenomicTransformer(nn.Module):
    def __init__(self, clinical_dim, genomic_dim):
        super().__init__()

        self.clinical_encoder = ClinicalEncoder(
            clinical_dim, 128, 0.3
        )

        self.genomic_encoder = GenomicEncoder(
            genomic_dim, 128, 0.3
        )

        self.fusion = ClinicalGenomicFusion(
            embed_dim=128,
            num_heads=4,
            num_layers=2,
            dropout=0.3,
        )

        self.survival_head = SurvivalPredictionHead(
            embed_dim=128,
            dropout=0.3,
        )

    def forward(self, clinical, genomic):
        clinical_token = self.clinical_encoder(clinical)
        genomic_token = self.genomic_encoder(genomic)

        fused = self.fusion(
            clinical_token,
            genomic_token,
        )

        risk = self.survival_head(fused)

        return {
            "survival": risk,
            "features": fused,
        }


model = ClinicalGenomicTransformer(
    clinical_dim=len(clinical_columns),
    genomic_dim=len(genomic_columns),
).to(DEVICE)

print(model)
print("\nModel initialized successfully.")
print("Device:", DEVICE)

ClinicalGenomicTransformer(
  (clinical_encoder): ClinicalEncoder(
    (encoder): Sequential(
      (0): Linear(in_features=20, out_features=128, bias=True)
      (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
    )
  )
  (genomic_encoder): GenomicEncoder(
    (encoder): Sequential(
      (0): Linear(in_features=200, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=256, out_features=128, bias=True)
      (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (fusion): ClinicalGenomicFusion(
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (line

##Test the model output

In [ ]:
# ============================================================
# CELL 6 — MODEL SHAPE TEST
# ============================================================

clinical_sample = torch.tensor(
    train[clinical_columns].to_numpy(dtype=np.float32)[:4],
    dtype=torch.float32,
    device=DEVICE,
)

genomic_sample = torch.tensor(
    train[genomic_columns].to_numpy(dtype=np.float32)[:4],
    dtype=torch.float32,
    device=DEVICE,
)

model.eval()

with torch.no_grad():
    output = model(clinical_sample, genomic_sample)

print("Clinical input:", clinical_sample.shape)
print("Genomic input:", genomic_sample.shape)
print("Risk output:", output["survival"].shape)
print("Fused representation:", output["features"].shape)

assert output["survival"].shape == (4, 1)
assert output["features"].shape == (4, 128)

print("\nModel shape test passed.")

Clinical input: torch.Size([4, 20])
Genomic input: torch.Size([4, 200])
Risk output: torch.Size([4, 1])
Fused representation: torch.Size([4, 128])

Model shape test passed.


##Create the pretraining dataset and DataLoader

In [ ]:

# ============================================================
# CLINICAL-GENOMIC PRETRAINING DATASET
# ============================================================

CLINICAL_DIM = len(clinical_columns)
GENOMIC_DIM = len(genomic_columns)

class ClinicalGenomicDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True).copy()

        self.clinical = self.data[clinical_columns].to_numpy(
            dtype=np.float32
        )
        self.genomic = self.data[genomic_columns].to_numpy(
            dtype=np.float32
        )
        self.time = self.data["survival_time"].to_numpy(
            dtype=np.float32
        )
        self.event = self.data["event"].to_numpy(
            dtype=np.float32
        )

        assert np.isfinite(self.clinical).all()
        assert np.isfinite(self.genomic).all()
        assert np.isfinite(self.time).all()
        assert (self.time > 0).all()
        assert np.isin(self.event, [0, 1]).all()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {
            "clinical": torch.from_numpy(self.clinical[idx]),
            "genomic": torch.from_numpy(self.genomic[idx]),
            "survival": torch.tensor(self.time[idx]),
            "event": torch.tensor(self.event[idx]),
        }


pretrain_dataset = ClinicalGenomicDataset(pool)

# Keep patient order fixed for the full-cohort Cox gradient method.
# Do NOT shuffle between its two gradient passes.
pretrain_loader = DataLoader(
    pretrain_dataset,
    batch_size=32,
    shuffle=False,
    drop_last=False,
    num_workers=0
)

print("Pretraining dataset:", len(pretrain_dataset))
print("Pretraining batches:", len(pretrain_loader))
print("Clinical dimension:", CLINICAL_DIM)
print("Genomic dimension:", GENOMIC_DIM)
print("Events:", int(pretrain_dataset.event.sum()))
print("Censored:", int((pretrain_dataset.event == 0).sum()))

batch = next(iter(pretrain_loader))
for key, value in batch.items():
    print(key, value.shape)


Pretraining dataset: 958
Pretraining batches: 30
Clinical dimension: 20
Genomic dimension: 200
Events: 59
Censored: 899
clinical torch.Size([32, 20])
genomic torch.Size([32, 200])
survival torch.Size([32])
event torch.Size([32])


##Check the full-cohort Cox objective

In [ ]:

# ============================================================
# FULL-COHORT COX LOSS AND GRADIENT COEFFICIENTS
# ============================================================

def full_cox_loss_and_coefficients(risk, survival_time, event):

    risk = risk.reshape(-1).double()
    time = survival_time.reshape(-1).double()
    event = event.reshape(-1).double()

    if not (risk.numel() == time.numel() == event.numel()):
        raise ValueError(
            "Risk, time and event lengths do not match."
        )

    if not torch.isfinite(risk).all():
        raise ValueError("Risk scores contain non-finite values.")

    if not torch.isfinite(time).all():
        raise ValueError("Survival times contain non-finite values.")

    if not torch.isfinite(event).all():
        raise ValueError("Event labels contain non-finite values.")

    if (time <= 0).any():
        raise ValueError("Survival times must be positive.")

    if not torch.all((event == 0) | (event == 1)):
        raise ValueError("Event labels must be 0 or 1.")

    n_events = event.sum()

    if n_events.item() == 0:
        raise ValueError(
            "The full training cohort has no observed events."
        )

    # Stabilize exponentials.
    risk_max = risk.max()
    z = torch.exp(risk - risk_max)

    # Patient j belongs to patient i's risk set when
    # patient j's observed time is >= patient i's time.
    risk_set = time.unsqueeze(0) >= time.unsqueeze(1)

    denominators = (
        risk_set.double() * z.unsqueeze(0)
    ).sum(dim=1).clamp_min(1e-12)

    log_denominators = torch.log(denominators) + risk_max

    loss = -(
        event * (risk - log_denominators)
    ).sum() / n_events

    # Gradient coefficients for the same full-cohort objective.
    # contributes = (
    #     time.unsqueeze(0) <= time.unsqueeze(1)
    # ).double()
    contributes = (
    time.unsqueeze(1) <= time.unsqueeze(0)
    ).double()

    event_weights = event / denominators

    coefficients = (
        -event
        + z * (
            contributes * event_weights.unsqueeze(1)
        ).sum(dim=0)
    ) / n_events

    return loss.detach().float(), coefficients.detach().float()


print("Full-cohort Cox function defined successfully.")


Full-cohort Cox function defined successfully.


In [ ]:

# ============================================================
# FULL-COHORT COX LOSS SANITY CHECK
# ============================================================

risk_test = torch.tensor(
    [0.2, -0.1, 0.7, 0.4],
    dtype=torch.float32
)

time_test = torch.tensor(
    [100.0, 200.0, 150.0, 300.0]
)

event_test = torch.tensor(
    [1.0, 0.0, 1.0, 0.0]
)

loss_test, coefficients_test = full_cox_loss_and_coefficients(
    risk_test,
    time_test,
    event_test
)

print("Test Cox loss:", loss_test.item())
print("Gradient coefficients:", coefficients_test)
print("Loss finite:", bool(torch.isfinite(loss_test)))
print("Coefficients finite:", bool(torch.isfinite(coefficients_test).all()))

assert torch.isfinite(loss_test)
assert torch.isfinite(coefficients_test).all()
assert coefficients_test.shape == risk_test.shape


Test Cox loss: 1.1562005281448364
Gradient coefficients: tensor([-0.3916,  0.1829, -0.0929,  0.3016])
Loss finite: True
Coefficients finite: True


##Verify the correction

In [ ]:

# ============================================================
# VERIFY COX GRADIENT AGAINST PYTORCH AUTOGRAD
# ============================================================

risk_test = torch.tensor(
    [0.2, -0.1, 0.7, 0.4],
    dtype=torch.float32,
    requires_grad=True
)

time_test = torch.tensor([100.0, 200.0, 150.0, 300.0])
event_test = torch.tensor([1.0, 0.0, 1.0, 0.0])

# Calculate loss and manual gradient coefficients
loss_test, coefficients_test = full_cox_loss_and_coefficients(
    risk_test.detach(),
    time_test,
    event_test
)

# Independently calculate the same Cox loss using autograd
r = risk_test.double()
t = time_test.double()
e = event_test.double()

z = torch.exp(r - r.max())
risk_set = t.unsqueeze(0) >= t.unsqueeze(1)

denominators = (
    risk_set.double() * z.unsqueeze(0)
).sum(dim=1).clamp_min(1e-12)

loss_autograd = -(
    e * (r - torch.log(denominators) - r.max())
).sum() / e.sum()

loss_autograd.backward()

print("Manual Cox loss:", loss_test.item())
print("Autograd Cox loss:", loss_autograd.item())
print("Manual coefficients:", coefficients_test)
print("Autograd gradients:", risk_test.grad)

gradient_match = torch.allclose(
    coefficients_test,
    risk_test.grad,
    atol=1e-5,
    rtol=1e-5
)

print("Gradient match:", gradient_match)

assert gradient_match, "Gradient check failed. Do not start training."
print("Gradient verification passed.")


Manual Cox loss: 1.1562005281448364
Autograd Cox loss: 1.156200490410435
Manual coefficients: tensor([-0.3916,  0.1829, -0.0929,  0.3016])
Autograd gradients: tensor([-0.3916,  0.1829, -0.0929,  0.3016])
Gradient match: True
Gradient verification passed.


In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)
print("Model device:", next(model.parameters()).device)

Device: cpu
Model device: cpu


In [ ]:
print("Model class:", type(model).__name__)
print("Forward signature:")
import inspect
print(inspect.signature(model.forward))

sample = next(iter(pretrain_loader))

model.eval()
with torch.no_grad():
    output = model(
        sample["clinical"].to(device),
        sample["genomic"].to(device)
    )

print("Output type:", type(output))

if isinstance(output, dict):
    for key, value in output.items():
        if torch.is_tensor(value):
            print(key, tuple(value.shape))
        else:
            print(key, type(value))
elif torch.is_tensor(output):
    print("Output shape:", tuple(output.shape))

Model class: ClinicalGenomicTransformer
Forward signature:
(clinical, genomic)
Output type: <class 'dict'>
survival (32, 1)
features (32, 128)


##Check the training setup

In [ ]:
# ============================================================
# PRETRAINING SETUP CHECK — BEFORE TRAINING
# ============================================================

import torch
import numpy as np
from torch.utils.data import DataLoader

# 1. Use the full 958-patient clinical-genomic pretraining pool.
pretrain_loader = DataLoader(
    pretrain_dataset,
    batch_size=32,
    shuffle=False,
    drop_last=False,
    num_workers=0
)

print("=" * 60)
print("PRETRAINING SETUP AUDIT")
print("=" * 60)

# 2. Check patient and batch counts.
n_patients = len(pretrain_dataset)
n_loaded = sum(
    len(batch["event"]) for batch in pretrain_loader
)

print("Pretraining patients:", n_patients)
print("Patients covered by loader:", n_loaded)
print("Number of batches:", len(pretrain_loader))
print("Batch size:", pretrain_loader.batch_size)
print("Shuffle:", pretrain_loader.sampler.__class__.__name__)
print("Drop last:", pretrain_loader.drop_last)

assert n_patients == 958, f"Expected 958 patients, got {n_patients}"
assert n_loaded == n_patients, "Some patients are missing from the loader"
assert pretrain_loader.drop_last is False
assert isinstance(model, ClinicalGenomicTransformer)

# 3. Check that the pool contains observed events.
event_count = int(pretrain_dataset.event.sum())
censored_count = n_patients - event_count

print("Observed events:", event_count)
print("Censored patients:", censored_count)

assert event_count > 0, "Cannot train Cox model without observed events"

# 4. Confirm that the model is on the selected device.
model = model.to(device)

print("Device:", device)
print("Model device:", next(model.parameters()).device)

# 5. Check the first batch.
batch = next(iter(pretrain_loader))

with torch.no_grad():
    output = model(
        batch["clinical"].to(device),
        batch["genomic"].to(device)
    )

risk = output["survival"].reshape(-1)

print("First batch risk shape:", tuple(risk.shape))
print("First batch survival shape:", tuple(batch["survival"].shape))
print("First batch event shape:", tuple(batch["event"].shape))
print("Risk scores finite:", bool(torch.isfinite(risk).all()))

assert risk.shape[0] == len(batch["event"])
assert torch.isfinite(risk).all()

print("\nPRETRAINING SETUP CHECK PASSED")

PRETRAINING SETUP AUDIT
Pretraining patients: 958
Patients covered by loader: 958
Number of batches: 30
Batch size: 32
Shuffle: SequentialSampler
Drop last: False
Observed events: 59
Censored patients: 899
Device: cpu
Model device: cpu
First batch risk shape: (32,)
First batch survival shape: (32,)
First batch event shape: (32,)
Risk scores finite: True

PRETRAINING SETUP CHECK PASSED


##Create the optimizer and test the full-cohort Cox objective

In [ ]:
print(
    "Coefficients finite:",
    bool(torch.isfinite(coefficients).all())
)

assert torch.isfinite(coefficients).all()

print("SANITY CHECK PASSED — NO WEIGHTS UPDATED")

Coefficients finite: True
SANITY CHECK PASSED — NO WEIGHTS UPDATED


##Add the two-pass training function

In [ ]:

# ============================================================
# TWO-PASS FULL-COHORT COX TRAINING FUNCTION
# ============================================================

def train_one_pretraining_epoch(model, loader, optimizer):
    model.train()

    all_risk = []
    all_survival = []
    all_events = []
    rng_states = []

    # --------------------------------------------------------
    # PASS A: Predict for every patient without storing graphs.
    # --------------------------------------------------------
    with torch.no_grad():
        for batch in loader:
            # Save RNG state immediately before this forward pass.
            rng_states.append({
                "cpu": torch.get_rng_state(),
                "cuda": (
                    torch.cuda.get_rng_state(device)
                    if torch.cuda.is_available()
                    else None
                )
            })

            clinical = batch["clinical"].to(device)
            genomic = batch["genomic"].to(device)

            output = model(clinical, genomic)
            risk = output["survival"].reshape(-1)

            all_risk.append(risk.detach().cpu())
            all_survival.append(batch["survival"].reshape(-1).cpu())
            all_events.append(batch["event"].reshape(-1).cpu())

    risks = torch.cat(all_risk)
    times = torch.cat(all_survival)
    events = torch.cat(all_events)

    if len(risks) != len(loader.dataset):
        raise RuntimeError("Pass A did not process every patient.")

    # Full-cohort Cox objective and its exact gradient coefficients.
    full_loss, coefficients = full_cox_loss_and_coefficients(
        risks, times, events
    )

    if not torch.isfinite(full_loss).all():
        raise RuntimeError("Non-finite full-cohort Cox loss.")

    if not torch.isfinite(coefficients).all():
        raise RuntimeError("Non-finite Cox gradient coefficients.")

    # --------------------------------------------------------
    # PASS B: Recompute predictions and accumulate gradients.
    # --------------------------------------------------------
    optimizer.zero_grad(set_to_none=True)
    offset = 0

    for batch_idx, batch in enumerate(loader):
        # Reproduce the corresponding Pass A dropout randomness.
        torch.set_rng_state(rng_states[batch_idx]["cpu"])

        if (
            torch.cuda.is_available()
            and rng_states[batch_idx]["cuda"] is not None
        ):
            torch.cuda.set_rng_state(
                rng_states[batch_idx]["cuda"], device
            )

        clinical = batch["clinical"].to(device)
        genomic = batch["genomic"].to(device)

        output = model(clinical, genomic)
        risk = output["survival"].reshape(-1)

        n = risk.numel()
        batch_coefficients = coefficients[offset:offset + n].to(device)

        # Surrogate objective: correct gradient at Pass A predictions
        # when the corresponding forward pass is reproduced.
        surrogate_loss = (risk * batch_coefficients).sum()
        surrogate_loss.backward()

        offset += n

    if offset != len(loader.dataset):
        raise RuntimeError(
            f"Pass B processed {offset} patients; "
            f"expected {len(loader.dataset)}."
        )

    # Reject invalid gradients before updating model parameters.
    for name, parameter in model.named_parameters():
        if parameter.grad is not None:
            if not torch.isfinite(parameter.grad).all():
                optimizer.zero_grad(set_to_none=True)
                raise RuntimeError(
                    f"Non-finite gradient detected in {name}."
                )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=1.0
    )

    optimizer.step()

    return {
        "train_loss": float(full_loss.item()),
        "patients": offset,
        "events": int(events.sum().item())
    }


print("Two-pass pretraining function defined.")


Two-pass pretraining function defined.


##Verify the two-pass gradient calculation

In [ ]:

# ============================================================
# VERIFY FULL-COHORT COX GRADIENTS AGAINST AUTOGRAD
# Independent mathematical test; does not train your model.
# ============================================================

import torch

# Four example patients with two observed events.
risk = torch.tensor(
    [0.2, -0.1, 0.7, 0.4],
    dtype=torch.float64,
    requires_grad=True
)

time = torch.tensor(
    [100.0, 200.0, 150.0, 300.0],
    dtype=torch.float64
)

event = torch.tensor(
    [1.0, 0.0, 1.0, 0.0],
    dtype=torch.float64
)

# Calculate the helper's loss and coefficients.
manual_loss, manual_coefficients = (
    full_cox_loss_and_coefficients(
        risk.detach(), time, event
    )
)

# Independently calculate the same Cox objective using autograd.
risk_set = time.unsqueeze(0) >= time.unsqueeze(1)
risk_max = risk.max()
z = torch.exp(risk - risk_max)

denominators = (
    risk_set.double() * z.unsqueeze(0)
).sum(dim=1).clamp_min(1e-12)

log_denominators = torch.log(denominators) + risk_max

autograd_loss = -(
    event * (risk - log_denominators)
).sum() / event.sum()

autograd_loss.backward()
autograd_gradient = risk.grad.detach().float()

print("Manual loss:", manual_loss.item())
print("Autograd loss:", autograd_loss.item())
print("Manual coefficients:", manual_coefficients)
print("Autograd gradient:", autograd_gradient)

loss_matches = torch.allclose(
    manual_loss,
    autograd_loss.detach().float(),
    atol=1e-5,
    rtol=1e-5
)

gradient_matches = torch.allclose(
    manual_coefficients,
    autograd_gradient,
    atol=1e-5,
    rtol=1e-5
)

print("Loss matches:", loss_matches)
print("Gradient matches:", gradient_matches)

assert loss_matches, "Cox loss verification failed."
assert gradient_matches, "Cox gradient verification failed."

print("COX MATHEMATICAL VERIFICATION PASSED")


Manual loss: 1.1562005281448364
Autograd loss: 1.1562004889447288
Manual coefficients: tensor([-0.3916,  0.1829, -0.0929,  0.3016])
Autograd gradient: tensor([-0.3916,  0.1829, -0.0929,  0.3016])
Loss matches: True
Gradient matches: True
COX MATHEMATICAL VERIFICATION PASSED


##Test the two-pass gradient accumulation

In [ ]:

# ============================================================
# TEST TWO-PASS GRADIENT ACCUMULATION ON A SMALL MODEL
# Does not modify the actual pretraining model.
# ============================================================

import copy
import torch
import torch.nn as nn

torch.manual_seed(123)

test_device = torch.device("cpu")

# Small deterministic model with dropout disabled.
class TinyCoxModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Linear(2, 1, bias=True)

    def forward(self, x):
        return self.net(x).reshape(-1)

# Four synthetic patients, split into batches of two.
x = torch.tensor([
    [0.2,  0.5],
    [0.8, -0.3],
    [0.4,  0.9],
    [-0.2, 0.1]
], dtype=torch.float32)

times = torch.tensor([100., 200., 150., 300.])
events = torch.tensor([1., 0., 1., 0.])

# ------------------------------------------------------------
# A. Reference gradient: one full-cohort forward/backward pass.
# ------------------------------------------------------------
reference_model = TinyCoxModel().to(test_device)
two_pass_model = copy.deepcopy(reference_model)

reference_model.zero_grad(set_to_none=True)

reference_risk = reference_model(x)
reference_loss, _ = full_cox_loss_and_coefficients(
    reference_risk.detach(), times, events
)

# Calculate the Cox loss directly with autograd.
r = reference_risk.double()
t = times.double()
e = events.double()

risk_set = t.unsqueeze(0) >= t.unsqueeze(1)
risk_max = r.max()
z = torch.exp(r - risk_max)

denominators = (
    risk_set.double() * z.unsqueeze(0)
).sum(dim=1).clamp_min(1e-12)

reference_autograd_loss = -(
    e * (r - torch.log(denominators) - risk_max)
).sum() / e.sum()

reference_autograd_loss.backward()

reference_grads = {
    name: parameter.grad.detach().clone()
    for name, parameter in reference_model.named_parameters()
}

# ------------------------------------------------------------
# B. Two-pass method: same model, batches of two.
# ------------------------------------------------------------
two_pass_model.zero_grad(set_to_none=True)

with torch.no_grad():
    risk_pass_a = []
    for start in range(0, len(x), 2):
        risk_pass_a.append(
            two_pass_model(x[start:start + 2]).detach()
        )

risk_pass_a = torch.cat(risk_pass_a)

_, coefficients = full_cox_loss_and_coefficients(
    risk_pass_a, times, events
)

offset = 0
for start in range(0, len(x), 2):
    batch_x = x[start:start + 2]
    batch_risk = two_pass_model(batch_x)

    n = len(batch_risk)
    batch_coefficients = coefficients[offset:offset + n]

    (batch_risk * batch_coefficients).sum().backward()
    offset += n

print("Reference Cox loss:", reference_autograd_loss.item())
print("Helper Cox loss:", reference_loss.item())

all_match = True

for name, parameter in two_pass_model.named_parameters():
    ref_grad = reference_grads[name]
    two_pass_grad = parameter.grad

    match = torch.allclose(
        ref_grad, two_pass_grad, atol=1e-5, rtol=1e-5
    )
    print(f"{name}: gradients match = {match}")

    if not match:
        all_match = False
        print("Reference:", ref_grad)
        print("Two-pass:", two_pass_grad)

assert all_match, "Two-pass gradient accumulation failed."
print("TWO-PASS GRADIENT TEST PASSED")


Reference Cox loss: 1.2340287094350173
Helper Cox loss: 1.234028697013855
net.weight: gradients match = True
net.bias: gradients match = True
TWO-PASS GRADIENT TEST PASSED


##Test dropout replay in my actual model

In [ ]:

# ============================================================
# VERIFY DROPOUT REPLAY FOR TWO-PASS TRAINING
# No optimizer step; model weights remain unchanged.
# ============================================================

import torch
import copy

# Save the original training/evaluation mode.
was_training = model.training

# Use a fixed batch from the pretraining loader.
sample = next(iter(pretrain_loader))
clinical = sample["clinical"].to(device)
genomic = sample["genomic"].to(device)

# Preserve CPU RNG state so this test does not disturb later randomness.
saved_cpu_rng = torch.get_rng_state()
saved_cuda_rng = (
    torch.cuda.get_rng_state(device)
    if torch.cuda.is_available()
    else None
)

try:
    model.train()

    # Pass A: record RNG state immediately before the forward pass.
    rng_cpu = torch.get_rng_state()
    rng_cuda = (
        torch.cuda.get_rng_state(device)
        if torch.cuda.is_available()
        else None
    )

    with torch.no_grad():
        output_a = model(clinical, genomic)["survival"].detach()

    # Pass B: restore the exact RNG state and repeat the same forward pass.
    torch.set_rng_state(rng_cpu)

    if torch.cuda.is_available() and rng_cuda is not None:
        torch.cuda.set_rng_state(rng_cuda, device)

    with torch.no_grad():
        output_b = model(clinical, genomic)["survival"].detach()

    max_difference = (output_a - output_b).abs().max().item()
    replay_matches = torch.allclose(
        output_a, output_b, atol=1e-7, rtol=1e-6
    )

    print("Dropout enabled:", model.training)
    print("Output shape:", tuple(output_a.shape))
    print("Maximum prediction difference:", max_difference)
    print("Dropout replay matches:", replay_matches)

    assert replay_matches, (
        "Dropout replay failed. Do not start two-pass training."
    )

    print("DROPOUT REPLAY TEST PASSED")

finally:
    # Restore the notebook's RNG state and original model mode.
    torch.set_rng_state(saved_cpu_rng)

    if torch.cuda.is_available() and saved_cuda_rng is not None:
        torch.cuda.set_rng_state(saved_cuda_rng, device)

    model.train(was_training)


Dropout enabled: True
Output shape: (32, 1)
Maximum prediction difference: 0.0
Dropout replay matches: True
DROPOUT REPLAY TEST PASSED


##Run a safe one-epoch smoke test

In [ ]:

# ============================================================
# ONE-EPOCH PRETRAINING SMOKE TEST
# Uses a temporary model; preserves the real model and RNG state.
# ============================================================

import copy
import math
import torch

# Confirm that the loader covers the full pool in sequential order.
assert len(pretrain_loader.dataset) == 958
assert len(pretrain_loader) == 30
assert pretrain_loader.drop_last is False
assert isinstance(
    pretrain_loader.sampler, torch.utils.data.SequentialSampler
)

# Preserve global random states.
saved_cpu_rng = torch.get_rng_state()
saved_cuda_rng = (
    torch.cuda.get_rng_state(device)
    if torch.cuda.is_available()
    else None
)

try:
    # Create an isolated copy of the current model.
    smoke_model = copy.deepcopy(model).to(device)

    smoke_optimizer = torch.optim.AdamW(
        smoke_model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    # Record parameters before the test update.
    parameters_before = {
        name: parameter.detach().clone()
        for name, parameter in smoke_model.named_parameters()
    }

    # Run exactly one full-cohort training epoch.
    result = train_one_pretraining_epoch(
        smoke_model,
        pretrain_loader,
        smoke_optimizer
    )

    # Validate the returned results.
    assert result["patients"] == 958, (
        f"Expected 958 patients, got {result['patients']}"
    )
    assert result["events"] == 59, (
        f"Expected 59 events, got {result['events']}"
    )
    assert math.isfinite(result["train_loss"]), (
        f"Non-finite training loss: {result['train_loss']}"
    )

    # Confirm the temporary model actually received an update.
    changed_parameters = [
        name
        for name, parameter in smoke_model.named_parameters()
        if not torch.equal(parameters_before[name], parameter.detach())
    ]

    assert len(changed_parameters) > 0, (
        "No parameters changed during the smoke test."
    )

    # Confirm all resulting parameters are finite.
    for name, parameter in smoke_model.named_parameters():
        assert torch.isfinite(parameter).all(), (
            f"Non-finite parameter detected: {name}"
        )

    print("=" * 60)
    print("ONE-EPOCH SMOKE TEST PASSED")
    print("=" * 60)
    print("Patients processed:", result["patients"])
    print("Observed events:", result["events"])
    print("Full-cohort Cox loss:", result["train_loss"])
    print("Parameters updated:", len(changed_parameters))
    print("All model parameters finite: True")
    print("Original model modified: False")

finally:
    # Restore RNG states so this test does not alter later randomness.
    torch.set_rng_state(saved_cpu_rng)

    if torch.cuda.is_available() and saved_cuda_rng is not None:
        torch.cuda.set_rng_state(saved_cuda_rng, device)


ONE-EPOCH SMOKE TEST PASSED
Patients processed: 958
Observed events: 59
Full-cohort Cox loss: 6.218307971954346
Parameters updated: 42
All model parameters finite: True
Original model modified: False


##Start clinical–genomic pretraining

In [ ]:

# ============================================================
# CLINICAL-GENOMIC PRETRAINING
# Full-cohort Cox loss + validation monitoring + early stopping
# ============================================================

import os
import copy
import math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# 1. Paths and settings
# ------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR, "best_clinical_genomic_pretraining.pt"
)
HISTORY_PATH = os.path.join(
    OUTPUT_DIR, "clinical_genomic_pretraining_history.csv"
)

MAX_EPOCHS = 100
PATIENCE = 15
MIN_DELTA = 1e-4

# Protect against accidentally starting over and overwriting results.
if os.path.exists(BEST_CHECKPOINT_PATH):
    raise FileExistsError(
        f"Checkpoint already exists: {BEST_CHECKPOINT_PATH}\n"
        "Inspect the existing checkpoint before starting another run."
    )

# ------------------------------------------------------------
# 2. Create a validation loader
# ------------------------------------------------------------
validation_df = pd.read_csv(VAL_PATH)
validation_dataset = ClinicalGenomicDataset(validation_df)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=32,
    shuffle=False,
    drop_last=False,
    num_workers=0
)

assert len(validation_dataset) == 49
assert int(validation_df["event"].sum()) == 12
assert len(validation_loader.dataset) == 49

print("Training patients:", len(pretrain_loader.dataset))
print("Training events:", int(pool["event"].sum()))
print("Validation patients:", len(validation_dataset))
print("Validation events:", int(validation_df["event"].sum()))
print("Device:", device)

# ------------------------------------------------------------
# 3. Validation: one full-cohort Cox loss
# ------------------------------------------------------------
def evaluate_full_cohort(model, loader):
    was_training = model.training
    model.eval()

    risks_all = []
    times_all = []
    events_all = []

    try:
        with torch.no_grad():
            for batch in loader:
                clinical = batch["clinical"].to(device)
                genomic = batch["genomic"].to(device)

                output = model(clinical, genomic)
                risks_all.append(
                    output["survival"].reshape(-1).detach().cpu()
                )
                times_all.append(batch["survival"].reshape(-1).cpu())
                events_all.append(batch["event"].reshape(-1).cpu())

        risks = torch.cat(risks_all)
        times = torch.cat(times_all)
        events = torch.cat(events_all)

        if len(risks) != len(loader.dataset):
            raise RuntimeError("Validation did not cover every patient.")

        loss, _ = full_cox_loss_and_coefficients(
            risks, times, events
        )

        if not torch.isfinite(loss).all():
            raise RuntimeError("Validation Cox loss is non-finite.")

        # Harrell-style C-index: larger risk should indicate
        # shorter survival. Only comparable event pairs count.
        risk_np = risks.numpy().reshape(-1)
        time_np = times.numpy().reshape(-1)
        event_np = events.numpy().reshape(-1)

        concordant = 0.0
        comparable = 0

        for i in range(len(time_np)):
            if event_np[i] != 1:
                continue

            for j in range(len(time_np)):
                if time_np[j] > time_np[i]:
                    comparable += 1

                    if risk_np[i] > risk_np[j]:
                        concordant += 1.0
                    elif risk_np[i] == risk_np[j]:
                        concordant += 0.5

        c_index = (
            concordant / comparable
            if comparable > 0
            else float("nan")
        )

        return {
            "loss": float(loss.item()),
            "c_index": float(c_index),
            "patients": len(risks),
            "events": int(events.sum().item()),
            "comparable_pairs": comparable
        }

    finally:
        model.train(was_training)

# ------------------------------------------------------------
# 4. Train and select the best checkpoint
# ------------------------------------------------------------
history = []
best_val_loss = float("inf")
best_epoch = None
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):

    train_result = train_one_pretraining_epoch(
        model,
        pretrain_loader,
        optimizer
    )

    val_result = evaluate_full_cohort(
        model,
        validation_loader
    )

    train_loss = train_result["train_loss"]
    val_loss = val_result["loss"]

    if not math.isfinite(train_loss) or not math.isfinite(val_loss):
        raise RuntimeError(
            f"Non-finite loss at epoch {epoch}: "
            f"train={train_loss}, validation={val_loss}"
        )

    improved = val_loss < (best_val_loss - MIN_DELTA)

    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state = copy.deepcopy(model.state_dict())

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": best_state,
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "validation_loss": val_loss,
                "validation_c_index": val_result["c_index"],
                "clinical_columns": clinical_columns,
                "genomic_columns": genomic_columns,
                "pool_patients": len(pretrain_loader.dataset),
                "pool_events": train_result["events"],
                "validation_patients": val_result["patients"],
                "validation_events": val_result["events"],
                "selection_metric": "full_cohort_validation_cox_loss"
            },
            BEST_CHECKPOINT_PATH
        )
    else:
        epochs_without_improvement += 1

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "validation_c_index": val_result["c_index"],
        "train_patients": train_result["patients"],
        "train_events": train_result["events"],
        "validation_patients": val_result["patients"],
        "validation_events": val_result["events"],
        "validation_comparable_pairs": val_result["comparable_pairs"],
        "best_checkpoint_updated": improved
    }
    history.append(record)

    # Save history after every epoch in case the session disconnects.
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Cox: {train_loss:.4f} | "
        f"Val Cox: {val_loss:.4f} | "
        f"Val C-index: {val_result['c_index']:.4f} | "
        f"Best epoch: {best_epoch}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"Early stopping: validation loss did not improve for "
            f"{PATIENCE} consecutive epochs."
        )
        break

# ------------------------------------------------------------
# 5. Restore the best validation-selected model
# ------------------------------------------------------------
if best_state is None:
    raise RuntimeError("No valid best checkpoint was produced.")

model.load_state_dict(best_state)
model.to(device)

print("\n" + "=" * 60)
print("CLINICAL-GENOMIC PRETRAINING FINISHED")
print("=" * 60)
print("Best epoch:", best_epoch)
print("Best validation Cox loss:", round(best_val_loss, 6))
print("Checkpoint:", BEST_CHECKPOINT_PATH)
print("Training history:", HISTORY_PATH)
print("Test set used for training/model selection: NO")


Training patients: 958
Training events: 59
Validation patients: 49
Validation events: 12
Device: cpu
Epoch 001 | Train Cox: 6.2183 | Val Cox: 2.7383 | Val C-index: 0.3488 | Best epoch: 1
Epoch 002 | Train Cox: 6.2079 | Val Cox: 2.7386 | Val C-index: 0.3837 | Best epoch: 1
Epoch 003 | Train Cox: 6.2036 | Val Cox: 2.7392 | Val C-index: 0.4031 | Best epoch: 1
Epoch 004 | Train Cox: 6.2059 | Val Cox: 2.7405 | Val C-index: 0.4225 | Best epoch: 1
Epoch 005 | Train Cox: 6.1435 | Val Cox: 2.7420 | Val C-index: 0.4380 | Best epoch: 1
Epoch 006 | Train Cox: 6.1321 | Val Cox: 2.7441 | Val C-index: 0.4419 | Best epoch: 1
Epoch 007 | Train Cox: 6.1023 | Val Cox: 2.7466 | Val C-index: 0.4729 | Best epoch: 1
Epoch 008 | Train Cox: 6.1003 | Val Cox: 2.7491 | Val C-index: 0.4922 | Best epoch: 1
Epoch 009 | Train Cox: 6.0763 | Val Cox: 2.7520 | Val C-index: 0.5233 | Best epoch: 1
Epoch 010 | Train Cox: 6.0888 | Val Cox: 2.7558 | Val C-index: 0.5310 | Best epoch: 1
Epoch 011 | Train Cox: 6.0312 | Val Cox

In [ ]:

# ============================================================
# AUDIT THE PRETRAINING RESULTS AND SAVED CHECKPOINT
# Read-only: no training and no test-set access.
# ============================================================

import os
import pandas as pd
import torch

history_path = os.path.join(
    OUTPUT_DIR,
    "clinical_genomic_pretraining_history.csv"
)
checkpoint_path = os.path.join(
    OUTPUT_DIR,
    "best_clinical_genomic_pretraining.pt"
)

assert os.path.exists(history_path), "Training history is missing."
assert os.path.exists(checkpoint_path), "Best checkpoint is missing."

history_check = pd.read_csv(history_path)

# Load only the checkpoint created by our own training run.
try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

print("=" * 65)
print("1. TRAINING HISTORY")
print("=" * 65)
print("Recorded epochs:", len(history_check))
print("Best validation-loss epoch:", int(
    history_check.loc[
        history_check["validation_loss"].idxmin(), "epoch"
    ]
))
print("Minimum validation Cox loss:", round(
    history_check["validation_loss"].min(), 6
))
print("Epoch with highest validation C-index:", int(
    history_check.loc[
        history_check["validation_c_index"].idxmax(), "epoch"
    ]
))
print("Highest validation C-index:", round(
    history_check["validation_c_index"].max(), 4
))

print("\nFirst five epochs:")
print(history_check.head(5).to_string(index=False))

print("\nLast five epochs:")
print(history_check.tail(5).to_string(index=False))

print("\n" + "=" * 65)
print("2. SAVED CHECKPOINT")
print("=" * 65)
print("Saved epoch:", checkpoint.get("epoch"))
print("Saved training loss:", checkpoint.get("train_loss"))
print("Saved validation loss:", checkpoint.get("validation_loss"))
print("Saved validation C-index:", checkpoint.get("validation_c_index"))
print("Training patients:", checkpoint.get("pool_patients"))
print("Training events:", checkpoint.get("pool_events"))
print("Validation patients:", checkpoint.get("validation_patients"))
print("Validation events:", checkpoint.get("validation_events"))
print("Selection metric:", checkpoint.get("selection_metric"))

state = checkpoint["model_state_dict"]
all_finite = all(
    torch.isfinite(value).all().item()
    for value in state.values()
    if torch.is_floating_point(value)
)

print("Checkpoint tensors finite:", all_finite)
print("Number of saved state entries:", len(state))

assert checkpoint["epoch"] == 1
assert checkpoint["pool_patients"] == 958
assert checkpoint["pool_events"] == 59
assert checkpoint["validation_patients"] == 49
assert checkpoint["validation_events"] == 12
assert all_finite

print("\nCHECKPOINT AUDIT PASSED")


1. TRAINING HISTORY
Recorded epochs: 16
Best validation-loss epoch: 1
Minimum validation Cox loss: 2.738344
Epoch with highest validation C-index: 15
Highest validation C-index: 0.5736

First five epochs:
 epoch  train_loss  validation_loss  validation_c_index  train_patients  train_events  validation_patients  validation_events  validation_comparable_pairs  best_checkpoint_updated
     1    6.218308         2.738344            0.348837             958            59                   49                 12                          258                     True
     2    6.207859         2.738608            0.383721             958            59                   49                 12                          258                    False
     3    6.203613         2.739183            0.403101             958            59                   49                 12                          258                    False
     4    6.205938         2.740472            0.422481             958    

##Inspect the pretrained model's parameter names

In [ ]:

# ============================================================
# INSPECT PRETRAINED MODEL WEIGHTS BEFORE TRANSFER
# Read-only audit
# ============================================================

import os
import torch

checkpoint_path = os.path.join(
    OUTPUT_DIR,
    "best_clinical_genomic_pretraining.pt"
)

try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

pretrained_state = checkpoint["model_state_dict"]

print("=" * 70)
print("PRETRAINED CHECKPOINT ARCHITECTURE AUDIT")
print("=" * 70)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Number of parameter/buffer entries:", len(pretrained_state))

print("\nAll saved state names and shapes:")
for name, tensor in pretrained_state.items():
    print(f"{name:65s} {tuple(tensor.shape)}")

print("\n" + "=" * 70)
print("ENCODER/FUSION/HEAD GROUPS")
print("=" * 70)

for prefix in ["clinical_encoder", "genomic_encoder",
               "fusion", "survival_head"]:
    matches = [
        name for name in pretrained_state
        if name.lower().startswith(prefix)
    ]
    print(f"\n{prefix}: {len(matches)} matching entries")
    for name in matches:
        print(" ", name, tuple(pretrained_state[name].shape))

print("\nNOTE:")
print("The printed prefixes are only a quick grouping check.")
print("The actual module names may differ in your implementation.")
print("Do not transfer weights until we compare both architectures.")


PRETRAINED CHECKPOINT ARCHITECTURE AUDIT
Checkpoint epoch: 1
Number of parameter/buffer entries: 42

All saved state names and shapes:
clinical_encoder.encoder.0.weight                                 (128, 20)
clinical_encoder.encoder.0.bias                                   (128,)
clinical_encoder.encoder.1.weight                                 (128,)
clinical_encoder.encoder.1.bias                                   (128,)
genomic_encoder.encoder.0.weight                                  (256, 200)
genomic_encoder.encoder.0.bias                                    (256,)
genomic_encoder.encoder.1.weight                                  (256,)
genomic_encoder.encoder.1.bias                                    (256,)
genomic_encoder.encoder.4.weight                                  (128, 256)
genomic_encoder.encoder.4.bias                                    (128,)
genomic_encoder.encoder.5.weight                                  (128,)
genomic_encoder.encoder.5.bias                     

##Inspect your original three-modality model

In [ ]:

# ============================================================
# INSPECT FINAL THREE-MODALITY TRANSFORMER
# READ-ONLY: does not train or modify model weights.
# ============================================================

import torch

# This assumes your instantiated final model is named `model`.
# If your model has a different variable name, replace `model` below.

print("=" * 70)
print("FINAL MODEL ARCHITECTURE")
print("=" * 70)

print("Model class:", type(model).__name__)

print("\nAll parameter and buffer names with shapes:")
final_state = model.state_dict()

for name, tensor in final_state.items():
    print(f"{name:75s} {tuple(tensor.shape)}")

print("\n" + "=" * 70)
print("MODULE NAMES")
print("=" * 70)

for name, module in model.named_modules():
    if name:
        print(f"{name:60s} {type(module).__name__}")

print("\n" + "=" * 70)
print("FINAL MODEL INSPECTION COMPLETE")
print("=" * 70)


FINAL MODEL ARCHITECTURE
Model class: ClinicalGenomicTransformer

All parameter and buffer names with shapes:
clinical_encoder.encoder.0.weight                                           (128, 20)
clinical_encoder.encoder.0.bias                                             (128,)
clinical_encoder.encoder.1.weight                                           (128,)
clinical_encoder.encoder.1.bias                                             (128,)
genomic_encoder.encoder.0.weight                                            (256, 200)
genomic_encoder.encoder.0.bias                                              (256,)
genomic_encoder.encoder.1.weight                                            (256,)
genomic_encoder.encoder.1.bias                                              (256,)
genomic_encoder.encoder.4.weight                                            (128, 256)
genomic_encoder.encoder.4.bias                                              (128,)
genomic_encoder.encoder.5.weight                 

In [ ]:
import os
import torch

CHECKPOINT_PATH = (
    "/content/gdrive/MyDrive/TCGA_BRCA/"
    "clinical_genomic_pretraining_leakage_free_v1/"
    "best_clinical_genomic_pretraining.pt"
)

print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))

assert os.path.exists(CHECKPOINT_PATH), (
    "Checkpoint not found. Check the Google Drive path."
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

print("\nCheckpoint loaded successfully.")
print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
        elif torch.is_tensor(value):
            print(f"{key}: tensor, shape={tuple(value.shape)}")
        else:
            print(f"{key}: {value}")

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint.get("state_dict", checkpoint)
    )

    print("\nMODEL PARAMETER NAMES AND SHAPES")
    for name, value in state_dict.items():
        if torch.is_tensor(value):
            print(f"{name:65s} {tuple(value.shape)}")

Checkpoint exists: True

Checkpoint loaded successfully.
Checkpoint type: <class 'dict'>

Top-level keys:
epoch: 1
model_state_dict: dictionary with 42 entries
optimizer_state_dict: dictionary with 2 entries
train_loss: 6.218307971954346
validation_loss: 2.738344430923462
validation_c_index: 0.3488372093023256
clinical_columns: ['years_to_birth', 'Tumor_purity', 'pathologic_stage', 'pathology_T_stage', 'pathology_N_stage', 'pathology_M_stage', 'number_of_lymph_nodes', 'radiation_therapy', 'histological_type_infiltratinglobularcarcinoma', 'histological_type_medullarycarcinoma', 'histological_type_metaplasticcarcinoma', 'histological_type_mixedhistology(pleasespecify)', 'histological_type_mucinouscarcinoma', 'histological_type_other,specify', 'PAM50_Her2', 'PAM50_LumA', 'PAM50_LumB', 'race_blackorafricanamerican', 'race_white', 'ethnicity_nothispanicorlatino']
genomic_columns: ['PSCA', 'ZNF385B', 'MYBPC1', 'CALML5', 'PRAME', 'FLT3', 'GREB1', 'TSPAN8', 'IL6ST', 'TUBA3E', 'RP1', 'DUSP9', '